# Fabric Database Hub hybrid SQL demo: live activity and analysis

Use this notebook during a demo to **simulate live activity** on the three SQL targets and to **analyze locks, blocking, and overall consumption** with the same telemetry that powers Database Hub **Performance**.

**How the simulation works.** The notebook can't reach the private SQL Servers and can't get an Azure Resource Manager token, so the activity runs on the servers themselves: the Contoso Care runner on T1 (driving T1 and T3) and in the T2 guest executes **activity profiles**. Every profile has a maximum duration and stops at auto-shutdown.

| Profile | What runs |
|---|---|
| `baseline` | A steady Contoso Care OLTP mix |
| `business-day` | The baseline plus scheduled, bounded incidents: blocking and long transactions on T1; CPU, IO, and missing-index bursts on T2; resource-governance and log-rate pressure on T3 |
| `incident:<scenario>` | One bounded incident on a chosen target: `locking-blocking`, `cpu-pressure`, `io-pressure`, `inefficient-query`, `missing-index`, `resource-governance`, `log-rate-pressure`, `blocking-chain`, `deadlock-storm`, `tempdb-contention`, `memory-grant-pressure`, `parallel-scans`, `log-flush-waits`, `slow-consumer`, or `capacity-max` |
| `off` | No demo activity |

**How this notebook changes the profile.** Only through the optional GitHub path: it triggers the `demo-control.yml` workflow with `workflow_dispatch`, using a GitHub fine-grained token that it reads from Key Vault at run time (prerequisite P9). Without P9, it prints the GitHub Actions link and the equivalent GitHub CLI command. Either way, it records the request in the Delta table `scenario_runs`, so the analysis can overlay scenario windows on the telemetry.

**Safety.** **Run all** is safe: the two cells that would trigger the workflow are flagged **Resource-changing** and only print a dry run unless `CONFIRM = True` (the stop cell also needs `STOP_ACTIVITY = True`). Tokens are never printed. The analysis is read-only and writes only to the lakehouse tables `scenario_runs`, `telemetry_snapshots`, `analysis_findings`, and `incident_windows`, which the Power BI history report reads.

> **Preview disclosure:** Database Hub, its Database Agent, and the SQL telemetry endpoint are previews; tables, columns, and behavior can change. Using a notebook `kusto` token against the telemetry endpoint isn't documented, so section 4 checks it and every query falls back to printing its KQL with the documented Azure Data Explorer web UI steps.

**Contents:** 1 Parameters · 2 Helpers · 3 Simulation control · 4 Telemetry access · 5 Estate pulse · 6 Locks and blocking · 7 Overall consumption · 8 Correlation · 9 See it in the Hub · 10 Optional Real-Time Dashboard · 11 Persist · 12 Stop and clean up · 13 Next steps

The companion notebook **`NB_DatabaseHub_Deployment`** validates the deployment and scripts the rest of the demo.

## 1. Parameters

The next cell is the **parameters cell** (in Fabric: **...** > **Toggle parameter cell**), so a pipeline notebook activity or `notebookutils.notebook.run` can override any value. It holds environment names, IDs, and the simulation request. **Never put a credential in it**: the optional GitHub token stays in Key Vault, and only the vault URL and the secret's *name* go here.

In [ ]:
# Parameters cell (tagged "parameters"). Environment values only: never put a credential here.

# Fabric workspace and default lakehouse (prerequisite P5).
# The repo stores no Fabric IDs (ADR 0021): the scripts' import fills these two in. Empty means this session's
# workspace and default lakehouse, so a notebook you import by hand works too.
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
LAKEHOUSE_NAME = "LH_DatabaseHub"

# Demo environment names (docs/architecture.md).
RESOURCE_GROUP = "rg-dbhub-demo"
T1_VM_NAME = "sqlvm01"
T2_MACHINE_NAME = "arcsql01"
T3_SERVER_PREFIX = "azuresql01-"
DEMO_DATABASE = "ContosoCare"
TARGETS = "T1,T2,T3"

# Analysis window: the last WINDOW_MINUTES, or an explicit UTC window inside the 7-day retention, for example
# WINDOW_START_UTC = "2026-10-20T14:00:00Z". BIN_MINUTES = 0 picks a bin size from the window length.
WINDOW_MINUTES = 60
WINDOW_START_UTC = ""
WINDOW_END_UTC = ""
BIN_MINUTES = 0
# Telemetry arrives 3-15 minutes after the activity; correlation looks this far past each scenario window.
TELEMETRY_LAG_MINUTES = 15

# Detected incident windows for the Power BI history report: consecutive bins over a threshold on one target. These are
# presenter heuristics, not product behavior (ADR 0014). Any blocked process, deadlock, or Log Rate Governor wait counts
# from one bin.
EPISODE_CPU_PERCENT = 80          # CPU at or above this (T3: percentage of the tier limit) for EPISODE_MIN_BINS bins
EPISODE_PAGEIOLATCH_SECONDS = 1   # Buffer IO (PAGEIOLATCH) waits per bin at or above this for EPISODE_MIN_BINS bins
EPISODE_WAIT_SECONDS = 5          # Buffer Latch, Memory, Parallelism, Tran Log IO, or Network IO waits per bin (Phase 25b)
EPISODE_MIN_BINS = 2

# Simulation request for demo-control.yml. PROFILE: baseline, business-day, off, or incident:<scenario>.
PROFILE = "baseline"
SIM_TARGET = "all"        # all, T1, T2, or T3 (an incident on "all" uses the scenario's default target)
DURATION_MINUTES = 30     # 1-240; every profile also stops at auto-shutdown
CONCURRENCY = 0           # 0 uses the scenario's default; otherwise 1-64
INTENSITY = 0             # 0 uses the scenario's default; otherwise 1-10
RESET_BEFORE = False
AUTO_CLEANUP = True

# Optional GitHub path: P7 for the workflow itself, P9 to trigger it from this notebook. The scripts' import sets
# GITHUB_REPOSITORY and GITHUB_ENVIRONMENT from github.repository and github.environment when they're set.
GITHUB_REPOSITORY = "MarcBushong/fabric-database-hub-demo"
GITHUB_REF = "main"
GITHUB_ENVIRONMENT = "demo"
GITHUB_WORKFLOW_FILE = "demo-control.yml"
KEY_VAULT_URL = ""                                  # P9, for example https://<vault-name>.vault.azure.net/ (empty: no trigger)
GITHUB_TOKEN_SECRET_NAME = "github-demo-control"    # the secret's name in Key Vault, not its value

# Append results to the default lakehouse's Delta tables (scenario_runs, telemetry_snapshots, analysis_findings,
# incident_windows) and prepare the five tables that the Power BI history report reads.
SAVE_RESULTS = True

# Safety switches. The resource-changing cells trigger demo-control.yml and do nothing unless CONFIRM is True.
# STOP_ACTIVITY must also be True for the stop cell in section 12, so "Run all" never stops what it just started.
CONFIRM = False
STOP_ACTIVITY = False

## 2. Helpers

Run the next three cells before any other code cell. They validate the parameters and define the helpers:

- **Tokens:** `notebookutils.credentials.getToken` with documented audience keys only (`pbi` for the Fabric REST API, `kusto` for the telemetry endpoint). There's no Azure Resource Manager audience. The optional GitHub token comes from `notebookutils.credentials.getSecret` for a single request and is never printed or kept.
- **Telemetry KQL** follows the telemetry article's rules: filter on `SampleTimeUTC` first; take per-series deltas of cumulative counters and skip intervals with a reset (never sum them); branch on `CounterType` (`65792` gauge, `272696576` cumulative, `537003264` percentage); group by `ResourceID` and `SqlServerInstanceName`; exclude the `Idle` and `Unknown` wait categories. For per-bin timelines it keeps the first and last sample of each series in each bin, the article's pattern for queries that must scale.
- **Results:** findings use the demo's status model (**Pass**, **Fail**, **Warning**, **Manual action required**, **Not supported**, **Not selected**).

In [ ]:
# Helpers, part 1: parameter checks, tokens, Fabric REST, and telemetry queries. Run the three helper cells first.
# Tokens come from notebookutils with documented audience keys only. A token is used in one HTTP header and never
# printed, logged, returned, or saved.
import datetime as dt
import hashlib
import json
import re
import time
import uuid

import pandas as pd
import requests

NOTEBOOK_NAME = "NB_DatabaseHub_LiveActivity"
FABRIC_API = "https://api.fabric.microsoft.com/v1"
TELEMETRY_QUERY_URL = "https://adx.centralus.arcdataservices.com/kusto/v1/rest/query"
TELEMETRY_CONNECTION_URI = "https://adx.centralus.arcdataservices.com/kusto/"
TELEMETRY_DATABASE = "ArcSqlTelemetry"
ADX_WEB_UI = "https://dataexplorer.azure.com/"
DATABASE_HUB_URL = "https://powerbi.com/workloads/fdh/databaseHub"
GITHUB_API = "https://api.github.com"
GITHUB_API_VERSION = "2026-03-10"
REPO_URL = "https://github.com/" + GITHUB_REPOSITORY
# notebookutils.credentials.getToken supports only these audience keys. None of them is Azure Resource Manager.
ALLOWED_TOKEN_AUDIENCES = ("storage", "pbi", "keyvault", "kusto")
STATUSES = ("Pass", "Fail", "Warning", "Manual action required", "Not supported", "Not selected")
PROFILES = ("baseline", "business-day", "off")
KNOWN_SCENARIOS = ("locking-blocking", "cpu-pressure", "io-pressure", "inefficient-query", "missing-index",
                   "resource-governance", "log-rate-pressure", "blocking-chain", "deadlock-storm", "tempdb-contention",
                   "memory-grant-pressure", "parallel-scans", "log-flush-waits", "slow-consumer", "capacity-max")
SCENARIO_DEFAULT_TARGET = {
    "locking-blocking": "T1", "cpu-pressure": "T2", "io-pressure": "T2", "inefficient-query": "T2", "missing-index": "T2",
    "resource-governance": "T3", "log-rate-pressure": "T3", "blocking-chain": "T1", "deadlock-storm": "T1",
    "tempdb-contention": "T2", "memory-grant-pressure": "T2", "parallel-scans": "T2", "log-flush-waits": "T1",
    "slow-consumer": "T3", "capacity-max": "T1",
}
SCENARIO_SIGNALS = {
    "locking-blocking": ("blocking",),
    "cpu-pressure": ("cpu",),
    "io-pressure": ("io",),
    "inefficient-query": ("cpu", "log_rate"),
    "missing-index": ("cpu", "io"),
    "resource-governance": ("cpu",),
    "log-rate-pressure": ("log_rate",),
    "blocking-chain": ("blocking",),
    "deadlock-storm": ("deadlock", "blocking"),
    "tempdb-contention": ("latch",),
    "memory-grant-pressure": ("memory",),
    "parallel-scans": ("parallelism", "cpu"),
    "log-flush-waits": ("tran_log",),
    "slow-consumer": ("network",),
    "capacity-max": ("cpu", "io"),
}
TARGET_COLORS = {"T1": "tab:blue", "T2": "tab:green", "T3": "tab:purple"}
DOCS = {
    "notebookutils": "https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials",
    "telemetry": "https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry",
    "monitor": "https://learn.microsoft.com/fabric/database/hub/monitor-sql",
    "dispatch": "https://docs.github.com/rest/actions/workflows#create-a-workflow-dispatch-event",
    "listItems": "https://learn.microsoft.com/rest/api/fabric/core/items/list-items",
    "rtd": "https://learn.microsoft.com/fabric/real-time-intelligence/dashboard-real-time-create",
}
TELEMETRY_HTTP_HINTS = {
    400: "HTTP 400: the KQL has an error, or a table or column changed during the preview (check names with getschema).",
    401: "HTTP 401: the endpoint didn't accept the notebook's kusto token. A notebook kusto token isn't documented for this endpoint.",
    403: "HTTP 403: the signed-in user needs Reader on the subscription, and Microsoft.Sql must be registered on it.",
    500: "HTTP 500: the request body wasn't valid.",
}
GITHUB_HTTP_HINTS = {
    401: "GitHub rejected the token: check that the Key Vault secret holds a valid fine-grained token.",
    403: "The token lacks the Actions (write) permission on this repository, or a policy blocks it.",
    404: "The repository or workflow wasn't found, or the token can't see the repository.",
    422: "The workflow rejected the ref or the inputs: check that demo-control.yml on GITHUB_REF defines these inputs.",
}
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
FINDINGS = []
SNAPSHOT_ROWS = []
SESSION_REQUESTS = []
FRAMES = {}
LAST_REQUEST = None
_DISPATCH_GUARD = {}
TELEMETRY_STATE = {"checked": False, "available": False, "message": "The telemetry access check (section 4) hasn't run yet."}


def _require(pattern, value, name):
    if not isinstance(value, str) or not re.fullmatch(pattern, value):
        raise ValueError("Parameter %s has an unexpected value: %r" % (name, value))
    return value


def _flag(value):
    """Parameters can arrive as strings from a pipeline; only an explicit true value counts as True."""
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ("true", "1", "yes")


CONFIRM = _flag(CONFIRM)
STOP_ACTIVITY = _flag(STOP_ACTIVITY)
SAVE_RESULTS = _flag(SAVE_RESULTS)
RESET_BEFORE = _flag(RESET_BEFORE)
AUTO_CLEANUP = _flag(AUTO_CLEANUP)
_GUID = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
# Empty IDs are resolved from the session once runtime_context() is defined, below (ADR 0021).
WORKSPACE_ID = str(WORKSPACE_ID or "").strip()
LAKEHOUSE_ID = str(LAKEHOUSE_ID or "").strip()
_require(r"[A-Za-z0-9 _.-]{1,256}", LAKEHOUSE_NAME, "LAKEHOUSE_NAME")
# Names are checked against strict patterns before they're placed in KQL or a GitHub request.
_require(r"[A-Za-z0-9._()-]{1,90}", RESOURCE_GROUP, "RESOURCE_GROUP")
_require(r"[A-Za-z0-9-]{1,64}", T1_VM_NAME, "T1_VM_NAME")
_require(r"[A-Za-z0-9._-]{1,64}", T2_MACHINE_NAME, "T2_MACHINE_NAME")
_require(r"[a-z0-9-]{1,63}", T3_SERVER_PREFIX, "T3_SERVER_PREFIX")
_require(r"[A-Za-z0-9_]{1,128}", DEMO_DATABASE, "DEMO_DATABASE")
_require(r"[A-Za-z0-9._-]+/[A-Za-z0-9._-]+", GITHUB_REPOSITORY, "GITHUB_REPOSITORY")
_require(r"[A-Za-z0-9._/-]{1,100}", GITHUB_REF, "GITHUB_REF")
_require(r"[A-Za-z0-9._-]{1,64}", GITHUB_ENVIRONMENT, "GITHUB_ENVIRONMENT")
_require(r"[A-Za-z0-9._-]+\.ya?ml", GITHUB_WORKFLOW_FILE, "GITHUB_WORKFLOW_FILE")
_require(r"[A-Za-z0-9-]{1,127}", GITHUB_TOKEN_SECRET_NAME, "GITHUB_TOKEN_SECRET_NAME")
KEY_VAULT_URL = str(KEY_VAULT_URL or "").strip()
if KEY_VAULT_URL:
    _require(r"https://[A-Za-z0-9-]{3,24}\.vault\.azure\.net/?", KEY_VAULT_URL, "KEY_VAULT_URL")
SELECTED_TARGETS = [item.strip().upper() for item in str(TARGETS).split(",") if item.strip()]
if not SELECTED_TARGETS or any(item not in ("T1", "T2", "T3") for item in SELECTED_TARGETS):
    raise ValueError("TARGETS must list T1, T2, and/or T3, for example 'T1,T2,T3'.")
if not 5 <= int(WINDOW_MINUTES) <= 7 * 24 * 60:
    raise ValueError("WINDOW_MINUTES must be between 5 minutes and the 7-day retention.")
if not 0 <= int(BIN_MINUTES) <= 60 or not 0 <= int(TELEMETRY_LAG_MINUTES) <= 60:
    raise ValueError("BIN_MINUTES and TELEMETRY_LAG_MINUTES must be between 0 and 60.")
EPISODE_CPU_PERCENT = float(EPISODE_CPU_PERCENT)
EPISODE_PAGEIOLATCH_SECONDS = float(EPISODE_PAGEIOLATCH_SECONDS)
EPISODE_MIN_BINS = int(EPISODE_MIN_BINS)
EPISODE_WAIT_SECONDS = float(EPISODE_WAIT_SECONDS)
if (not 0 < EPISODE_CPU_PERCENT <= 100 or EPISODE_PAGEIOLATCH_SECONDS <= 0 or not 1 <= EPISODE_MIN_BINS <= 60
        or EPISODE_WAIT_SECONDS <= 0):
    raise ValueError("EPISODE_CPU_PERCENT must be above 0 and at most 100, EPISODE_PAGEIOLATCH_SECONDS and "
                     "EPISODE_WAIT_SECONDS above 0, and EPISODE_MIN_BINS between 1 and 60.")

_REDACTIONS = (
    (re.compile(r"eyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "[REDACTED:jwt]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/=-]{8,}"), r"\1[REDACTED]"),
    (re.compile(r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})\b"), "[REDACTED:github-token]"),
    (re.compile(r"(?i)((?:^|[?&;\s\"'])sig=)[^&\s\"']+"), r"\1[REDACTED]"),
)


def redact(value):
    """Removes token-like strings from text before it's printed or saved."""
    text = "" if value is None else str(value)
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def utc_now():
    return dt.datetime.now(dt.timezone.utc)


def _notebookutils():
    try:
        return notebookutils  # noqa: F821 - Fabric provides notebookutils in every notebook session.
    except NameError:
        try:
            import notebookutils as module
            return module
        except ImportError:
            return None


def _spark():
    try:
        return spark  # noqa: F821 - Fabric provides the Spark session.
    except NameError:
        return None


def display_frame(frame):
    try:
        display(frame)  # noqa: F821 - Fabric provides display().
    except NameError:
        print(frame.to_string(index=False))


def get_token(audience):
    """Returns a token for a documented notebookutils audience key. Never print, log, or save the result."""
    if audience not in ALLOWED_TOKEN_AUDIENCES:
        raise ValueError("notebookutils.credentials.getToken supports only %s." % ", ".join(ALLOWED_TOKEN_AUDIENCES))
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    return utils.credentials.getToken(audience)


def runtime_context():
    utils = _notebookutils()
    if utils is None:
        return {}
    try:
        return dict(utils.runtime.context or {})
    except Exception:
        return {}


# The repo stores no Fabric IDs (ADR 0021). Empty IDs, as in a notebook imported by hand, are this session's own
# workspace and default lakehouse; the scripts' import fills in the ones it resolved by name.
if not WORKSPACE_ID or not LAKEHOUSE_ID:
    _session = runtime_context()
    WORKSPACE_ID = WORKSPACE_ID or str(_session.get("currentWorkspaceId") or "")
    LAKEHOUSE_ID = LAKEHOUSE_ID or str(_session.get("defaultLakehouseId") or "")
if WORKSPACE_ID:
    _require(_GUID, WORKSPACE_ID, "WORKSPACE_ID")
if LAKEHOUSE_ID:
    _require(_GUID, LAKEHOUSE_ID, "LAKEHOUSE_ID")


def _retry_after(response):
    try:
        return min(max(int(response.headers.get("Retry-After", "5")), 1), 60)
    except ValueError:
        return 5


def _error_text(response):
    try:
        body = response.json()
    except ValueError:
        return redact((response.text or "")[:300])
    if isinstance(body, dict):
        parts = [str(body.get(key)) for key in ("errorCode", "message", "requestId", "documentation_url") if body.get(key)]
        if isinstance(body.get("error"), dict):
            parts += [str(body["error"].get(key)) for key in ("code", "message") if body["error"].get(key)]
        return redact(" ".join(parts)[:300])
    return ""


def fabric_get(path_or_url):
    """GET from the Fabric REST API with the notebook user's pbi token. Returns (status, body, error)."""
    url = path_or_url if path_or_url.startswith(FABRIC_API + "/") else FABRIC_API + path_or_url
    if not url.startswith(FABRIC_API + "/"):
        raise ValueError("Refusing to send a Fabric token outside " + FABRIC_API)
    response = None
    for attempt in range(4):
        try:
            response = requests.get(url, headers={"Authorization": "Bearer " + get_token("pbi")}, timeout=60)
        except requests.RequestException as error:
            return 0, None, redact(error)
        except Exception as error:
            return 0, None, "notebookutils didn't return a pbi token: " + redact(error)
        if response.status_code == 429 and attempt < 3:
            time.sleep(_retry_after(response))
            continue
        break
    if response.ok:
        try:
            return response.status_code, response.json(), ""
        except ValueError:
            return response.status_code, None, "The response wasn't JSON."
    return response.status_code, None, _error_text(response)


def fabric_list_items(workspace_id, item_type):
    """Lists items of one type in a workspace, following continuation pages. Returns (items, error)."""
    items = []
    url = "%s/workspaces/%s/items?type=%s" % (FABRIC_API, workspace_id, item_type)
    for _ in range(50):
        if not url:
            return items, ""
        status, body, error = fabric_get(url)
        if status != 200 or body is None:
            return None, "HTTP %s %s" % (status, error)
        # A continuation URI can drop the type filter, so keep only items of the requested type.
        items.extend(item for item in body.get("value", []) if item.get("type") in (None, item_type))
        url = body.get("continuationUri")
        if url and not url.startswith(FABRIC_API + "/"):
            return None, "Unexpected continuation URI; stopped so the token stays inside the Fabric API."
    return (items, "") if not url else (None, "Stopped after 50 pages of results.")


# ---- Telemetry ----------------------------------------------------------------------------------------------

def _parse_utc(text):
    value = pd.Timestamp(str(text).strip())
    value = value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    return value.to_pydatetime()


def analysis_window():
    """Returns (start, end, bin_minutes) for the analysis, inside the telemetry's 7-day retention."""
    now = utc_now()
    end = _parse_utc(WINDOW_END_UTC) if str(WINDOW_END_UTC).strip() else now
    end = min(end, now)
    start = _parse_utc(WINDOW_START_UTC) if str(WINDOW_START_UTC).strip() else end - dt.timedelta(minutes=int(WINDOW_MINUTES))
    if start >= end:
        raise ValueError("The analysis window must start before it ends.")
    if now - start > dt.timedelta(days=7):
        print("Note: the window starts more than 7 days ago; telemetry retention is up to 7 days, so older data isn't available.")
    span_minutes = (end - start).total_seconds() / 60
    bin_minutes = int(BIN_MINUTES) or (1 if span_minutes <= 120 else 5 if span_minutes <= 720 else 15)
    return start, end, bin_minutes


def kql_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kql_datetime(value):
    return "datetime(" + value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") + ")"


def kql_prelude(start, end, bin_minutes=1):
    """Time window, bin, names, and a function that maps telemetry rows to T1, T2, or T3 (empty for anything else)."""
    return "\n".join([
        "let _start = %s;" % kql_datetime(start),
        "let _end = %s;" % kql_datetime(end),
        "let _bin = %dm;" % int(bin_minutes),
        "let _rg = %s;" % kql_string(RESOURCE_GROUP),
        "let _db = %s;" % kql_string(DEMO_DATABASE),
        "let _t1 = %s;" % kql_string(T1_VM_NAME),
        "let _t2 = %s;" % kql_string(T2_MACHINE_NAME),
        "let _t3prefix = %s;" % kql_string(T3_SERVER_PREFIX),
        "let _t3suffix = %s;" % kql_string("/" + DEMO_DATABASE),
        "let _target = (resourceType: string, resourceName: string) {",
        "    case(resourceType == 'virtualMachines' and resourceName =~ _t1, 'T1',",
        "         resourceType == 'machines' and resourceName =~ _t2, 'T2',",
        "         resourceType == 'servers/databases' and resourceName startswith _t3prefix and resourceName endswith _t3suffix, 'T3',",
        "         '')",
        "};",
    ])


def _to_utc(series):
    try:
        return pd.to_datetime(series, utc=True, format="ISO8601")
    except (TypeError, ValueError):
        return pd.to_datetime(series, utc=True, errors="coerce")


def kusto_frame(body):
    """Turns a Kusto v1 response (Tables[0].Columns and Rows) into a DataFrame, mapping values by column name."""
    tables = body.get("Tables") or []
    if not tables:
        return pd.DataFrame()
    first = tables[0]
    columns = first.get("Columns", [])
    frame = pd.DataFrame(first.get("Rows", []), columns=[column.get("ColumnName") for column in columns])
    for column in columns:
        if str(column.get("ColumnType") or column.get("DataType")).lower() in ("datetime", "system.datetime"):
            frame[column["ColumnName"]] = _to_utc(frame[column["ColumnName"]])
    return frame


def _post_kql(kql):
    """Sends read-only KQL to the telemetry endpoint. Returns (status, frame, message)."""
    try:
        token = get_token("kusto")
    except Exception as error:
        return 0, None, "notebookutils didn't return a kusto token: " + redact(error)
    try:
        response = requests.post(
            TELEMETRY_QUERY_URL,
            json={"db": TELEMETRY_DATABASE, "csl": str(kql)},
            headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"},
            timeout=180,
        )
    except requests.RequestException as error:
        return 0, None, "The telemetry endpoint couldn't be reached: " + redact(error)
    finally:
        token = None
    if response.status_code == 200:
        try:
            return 200, kusto_frame(response.json()), ""
        except ValueError:
            return 0, None, "The telemetry endpoint returned something other than Kusto v1 JSON."
    hint = TELEMETRY_HTTP_HINTS.get(response.status_code, "HTTP %d." % response.status_code)
    return response.status_code, None, (hint + " " + _error_text(response)).strip()


def print_kql_fallback(kql, title, reason):
    print("[%s] Not queried from the notebook: %s" % (title, redact(reason)))
    print("Run the KQL below yourself with the documented Azure Data Explorer web UI:")
    print("  1. Open %s and sign in with an account that can read the demo subscription." % ADX_WEB_UI)
    print("  2. Select Add > Connection, and enter the connection URI %s" % TELEMETRY_CONNECTION_URI)
    print("  3. Expand the connection, select the %s database, paste the query, and select Run." % TELEMETRY_DATABASE)
    print("----- KQL: %s -----" % title)
    print(kql)
    print("----- end of KQL -----")


def query_telemetry(kql, title):
    """Runs KQL with the notebook's kusto token. Returns a DataFrame, or None after printing the fallback."""
    if TELEMETRY_STATE["checked"] and not TELEMETRY_STATE["available"]:
        print_kql_fallback(kql, title, TELEMETRY_STATE["message"])
        return None
    status, frame, message = _post_kql(kql)
    if status == 200:
        return frame
    print_kql_fallback(kql, title, message)
    return None

In [ ]:
# Helpers, part 2: analysis results, findings, snapshots, and charts. Run it after part 1.


def _safe_number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return 0.0
    return 0.0 if number != number else number


def to_series(frame, value_column, how="sum", where=None):
    """One value per target and bin, ready to chart: columns Target, Bucket, Value."""
    if frame is None or frame.empty:
        return None
    part = frame if where is None else frame[where(frame)]
    if part.empty:
        return None
    grouped = part.groupby(["Target", "Bucket"], as_index=False)[value_column].agg(how)
    return grouped.rename(columns={value_column: "Value"})


def _naive_utc(value):
    stamp = pd.Timestamp(value)
    return (stamp.tz_convert("UTC") if stamp.tzinfo is not None else stamp).tz_localize(None)


def plot_panels(title, panels, windows=()):
    """Draws one panel per (name, series, unit), one line per target, with scenario windows shaded."""
    panels = [(name, series, unit) for name, series, unit in panels if series is not None and not series.empty]
    if not panels:
        print("%s: nothing to chart." % title)
        return
    try:
        import matplotlib.dates as mdates
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib isn't available, so the charts are skipped.")
        return
    figure, axes = plt.subplots(len(panels), 1, figsize=(12, 2.6 * len(panels) + 0.8), sharex=True, squeeze=False)
    for index, (axis, (name, series, unit)) in enumerate(zip(axes[:, 0], panels)):
        for target in sorted(series["Target"].unique()):
            line = series[series["Target"] == target].sort_values("Bucket")
            axis.plot([_naive_utc(value) for value in line["Bucket"]], line["Value"], marker=".", linewidth=1.4,
                      label=target, color=TARGET_COLORS.get(target))
        for window in windows:
            axis.axvspan(_naive_utc(window["start"]), _naive_utc(window["end"]), color="tab:orange", alpha=0.12)
            if index == 0:
                axis.text(_naive_utc(window["start"]), 0.98, " " + window["label"], transform=axis.get_xaxis_transform(),
                          fontsize=7, va="top", color="tab:orange")
        axis.set_title(name, loc="left", fontsize=10)
        axis.set_ylabel(unit)
        axis.grid(alpha=0.3)
        axis.legend(loc="upper right", fontsize=8)
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[-1, 0].set_xlabel("UTC (shaded: scenario windows from scenario_runs)")
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()


def add_finding(analysis, target, status, finding, key, evidence=None, request_id=None, start=None, end=None):
    """Records one finding with the ADR 0012 status model. Re-running replaces a finding with the same key."""
    if status not in STATUSES:
        raise ValueError("Unknown status: " + status)
    finding_id = "%s.%s.%s" % (analysis, target, key)
    record = {
        "run_id": RUN_ID, "finding_id": finding_id, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": utc_now(),
        "analysis": analysis, "target": target, "status": status, "finding": redact(finding),
        "evidence": redact(json.dumps(evidence or {}, default=str, sort_keys=True)), "request_id": request_id,
        "window_start_utc": start, "window_end_utc": end,
    }
    FINDINGS[:] = [item for item in FINDINGS if item["finding_id"] != finding_id]
    FINDINGS.append(record)
    print("%-24s [%s] %s" % (status, target, record["finding"]))
    return record


def add_snapshot(analysis, frame, start, end, metrics=None, metric_column=None, value_column=None, units=None):
    """Keeps analysis results in long form for telemetry_snapshots. Re-running replaces that analysis's rows."""
    SNAPSHOT_ROWS[:] = [row for row in SNAPSHOT_ROWS if row["analysis"] != analysis]
    if frame is None or frame.empty:
        return
    captured = utc_now()
    units = units or {}
    for record in frame.to_dict("records"):
        pairs = [(record.get(metric_column), record.get(value_column))] if metric_column else [(metric, record.get(metric)) for metric in metrics]
        for metric, value in pairs:
            if value is None or pd.isna(value):
                continue
            SNAPSHOT_ROWS.append({
                "run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": captured, "analysis": analysis,
                "target": record.get("Target"), "resource_type": record.get("ResourceType"), "resource_name": record.get("ResourceName"),
                "instance_name": record.get("SqlServerInstanceName"), "database_name": record.get("DatabaseName"),
                "metric": str(metric), "bucket_utc": record.get("Bucket"), "value": float(value), "unit": units.get(metric, ""),
                "window_start_utc": start, "window_end_utc": end,
            })


# Signal families: the expected signal of each scenario (SCENARIO_SIGNALS) and the incident windows' signal_family.
SIGNAL_LABELS = {"blocking": "Blocking", "cpu": "CPU pressure", "io": "IO pressure", "log_rate": "Log-rate pressure",
                 "deadlock": "Deadlocks", "latch": "tempdb latch contention", "memory": "Memory grant pressure",
                 "parallelism": "Parallelism waits", "tran_log": "Log flush waits", "network": "Slow client (network waits)",
                 "activity": "Activity"}
SIGNAL_UNITS = {"blocking": "processes", "cpu": "%", "io": "s", "log_rate": "s", "deadlock": "deadlocks", "latch": "s",
                "memory": "s", "parallelism": "s", "tran_log": "s", "network": "s", "activity": "requests"}
# The SqlServerWaitStats category behind each wait family (the telemetry article's WaitCategory values).
WAIT_FAMILY_CATEGORIES = {"log_rate": "Log Rate Governor", "latch": "Buffer Latch", "memory": "Memory",
                          "parallelism": "Parallelism", "tran_log": "Tran Log IO", "network": "Network IO"}
# Detected-episode rules for incident_windows: (signal family, threshold, strictly above it, minimum bins).
EPISODE_RULES = (
    ("blocking", 0.0, True, 1),
    ("cpu", EPISODE_CPU_PERCENT, False, EPISODE_MIN_BINS),
    ("io", EPISODE_PAGEIOLATCH_SECONDS, False, EPISODE_MIN_BINS),
    ("log_rate", 0.0, True, 1),
    ("deadlock", 0.0, True, 1),
    ("latch", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("memory", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("parallelism", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("tran_log", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("network", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
)


def expected_signals(profile):
    if profile.startswith("incident:"):
        scenario = profile.split(":", 1)[1]
        if scenario in SCENARIO_SIGNALS:
            return SCENARIO_SIGNALS[scenario]
        keywords = (("lock", "blocking"), ("block", "blocking"), ("cpu", "cpu"), ("io", "io"), ("log", "log_rate"), ("index", "cpu"))
        return tuple(sorted({signal for word, signal in keywords if word in scenario})) or ("activity",)
    return ("activity",)


def signal_episodes(series, threshold, bin_minutes, min_bins, strict=False):
    """Groups consecutive bins at or above a threshold (strictly above it with strict=True) into episodes, per target.
    series has the columns Target, Bucket, and Value (see to_series). Episodes shorter than min_bins bins are dropped;
    each ends one bin after its last bin. Returns dicts with target, start, last, end, peak, and bins."""
    if series is None or series.empty:
        return []
    above = series[series["Value"] > threshold] if strict else series[series["Value"] >= threshold]
    above = above.groupby(["Target", "Bucket"], as_index=False)["Value"].max().sort_values(["Target", "Bucket"])
    step = pd.Timedelta(minutes=int(bin_minutes))
    episodes = []
    for target, group in above.groupby("Target"):
        current = None
        for _, row in group.iterrows():
            if current is not None and row["Bucket"] - current["last"] <= step:
                current["last"] = row["Bucket"]
                current["peak"] = max(current["peak"], float(row["Value"]))
                current["bins"] += 1
            else:
                if current is not None:
                    episodes.append(current)
                current = {"target": target, "start": row["Bucket"], "last": row["Bucket"], "peak": float(row["Value"]), "bins": 1}
        if current is not None:
            episodes.append(current)
    kept = [episode for episode in episodes if episode["bins"] >= int(min_bins)]
    for episode in kept:
        episode["end"] = episode["last"] + step
    return kept


def episode_series(family):
    """The per-target, per-bin series that the episode rules scan, from the frames that sections 6 and 7 queried."""
    if family == "blocking":
        return to_series(FRAMES.get("gauges"), "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
    if family == "cpu":
        return to_series(FRAMES.get("cpu"), "AvgCPU", "max")
    if family == "deadlock":
        return to_series(FRAMES.get("rates"), "Delta", "sum", lambda f: f["CounterName"] == "Number of Deadlocks/sec")
    waits = FRAMES.get("waits")
    if family == "io":
        series = to_series(waits, "WaitMs", "sum", lambda f: f["PageIoLatch"].astype(bool))
    elif family in WAIT_FAMILY_CATEGORIES:
        category = WAIT_FAMILY_CATEGORIES[family]
        series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == category)
    else:
        return None
    if series is not None:
        series["Value"] = series["Value"] / 1000.0
    return series


In [ ]:
# Helpers, part 3: simulation control (demo-control.yml through GitHub, P9) and Delta tables. Run it after part 2.


def build_simulation_request(profile, target, duration_minutes, concurrency, intensity, reset_before, auto_cleanup):
    """Validates one request and builds the demo-control.yml inputs. No network calls."""
    profile = str(profile).strip()
    scenario = None
    if profile.startswith("incident:"):
        scenario = profile.split(":", 1)[1]
        if not re.fullmatch(r"[a-z0-9][a-z0-9-]{0,47}", scenario):
            raise ValueError("incident:<scenario> needs a scenario ID such as incident:locking-blocking.")
        if scenario not in KNOWN_SCENARIOS:
            print("Note: '%s' isn't one of the documented scenarios (%s); demo-control.yml decides whether it exists." % (scenario, ", ".join(KNOWN_SCENARIOS)))
    elif profile not in PROFILES:
        raise ValueError("PROFILE must be baseline, business-day, off, or incident:<scenario>.")
    target = str(target).strip()
    target = "all" if target.lower() == "all" else target.upper()
    if target not in ("all", "T1", "T2", "T3"):
        raise ValueError("SIM_TARGET must be all, T1, T2, or T3.")
    if scenario and target == "all":
        if scenario not in SCENARIO_DEFAULT_TARGET:
            raise ValueError("Set SIM_TARGET to T1, T2, or T3 for incident:%s." % scenario)
        target = SCENARIO_DEFAULT_TARGET[scenario]
        print("An incident runs on one target: using %s, the default for %s." % (target, scenario))
    duration = int(duration_minutes)
    concurrency = int(concurrency)
    intensity = int(intensity)
    if not 1 <= duration <= 240:
        raise ValueError("DURATION_MINUTES must be between 1 and 240.")
    if not 0 <= concurrency <= 64:
        raise ValueError("CONCURRENCY must be 0 (scenario default) or between 1 and 64.")
    if not 0 <= intensity <= 10:
        raise ValueError("INTENSITY must be 0 (scenario default) or between 1 and 10.")
    # Input names follow ADR 0011; demo-control.yml declares them. Values are strings, as workflow_dispatch expects.
    inputs = {
        "environment": GITHUB_ENVIRONMENT,
        "target": target,
        "scenario": profile,
        "duration": str(duration),
        "reset-before": "true" if _flag(reset_before) else "false",
        "auto-cleanup": "true" if _flag(auto_cleanup) else "false",
    }
    if concurrency:
        inputs["concurrency"] = str(concurrency)
    if intensity:
        inputs["intensity"] = str(intensity)
    return {
        "profile": profile, "scenario": scenario, "target": target,
        "targets": ["T1", "T2", "T3"] if target == "all" else [target],
        "duration_minutes": duration, "concurrency": concurrency or None, "intensity": intensity or None,
        "reset_before": _flag(reset_before), "auto_cleanup": _flag(auto_cleanup), "inputs": inputs,
        "fingerprint": hashlib.sha256(json.dumps(inputs, sort_keys=True).encode("utf-8")).hexdigest()[:16],
    }


def gh_cli_command(request):
    parts = ["gh", "workflow", "run", GITHUB_WORKFLOW_FILE, "--repo", GITHUB_REPOSITORY, "--ref", GITHUB_REF]
    for name in sorted(request["inputs"]):
        parts += ["-f", "%s=%s" % (name, request["inputs"][name])]
    return " ".join(parts)


def print_manual_trigger(request):
    print("Trigger it yourself (GitHub path, needs P7):")
    print("  1. Open %s/actions/workflows/%s and select Run workflow." % (REPO_URL, GITHUB_WORKFLOW_FILE))
    print("  2. Use branch %s and these inputs: %s" % (GITHUB_REF, ", ".join("%s=%s" % item for item in sorted(request["inputs"].items()))))
    print("  Or with the GitHub CLI:")
    print("  " + gh_cli_command(request))
    print("  Or locally with PowerShell 7 from the repo: ./scripts/start-demo.ps1 (see docs/deployment.md).")


def print_simulation_plan(request):
    end = "the request time" if request["profile"] == "off" else "%d minutes later" % request["duration_minutes"]
    print("Profile:  %s" % request["profile"])
    print("Targets:  %s" % ", ".join(request["targets"]))
    print("Window:   from the request time to %s, or auto-shutdown, whichever comes first" % end)
    print("Workflow: POST %s/repos/%s/actions/workflows/%s/dispatches" % (GITHUB_API, GITHUB_REPOSITORY, GITHUB_WORKFLOW_FILE))
    print("Body:     " + json.dumps({"ref": GITHUB_REF, "inputs": request["inputs"], "return_run_details": True}, sort_keys=True))
    if KEY_VAULT_URL:
        print("P9:       configured. The token is read from Key Vault (%s, secret %s) only when the request is sent." % (KEY_VAULT_URL, GITHUB_TOKEN_SECRET_NAME))
    else:
        print("P9:       not configured, so the notebook prints the link and command instead of sending the request.")
        print_manual_trigger(request)


def _github_request(method, url, body=None):
    """One GitHub REST call with the P9 token from Key Vault. The token is never printed or kept."""
    if not url.startswith("%s/repos/%s/actions/" % (GITHUB_API, GITHUB_REPOSITORY)):
        raise ValueError("Refusing to send the GitHub token outside this repository's Actions API.")
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    headers = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": GITHUB_API_VERSION}
    headers["Authorization"] = "Bearer " + utils.credentials.getSecret(KEY_VAULT_URL, GITHUB_TOKEN_SECRET_NAME)
    try:
        return requests.request(method, url, json=body, headers=headers, timeout=30)
    finally:
        headers.clear()


def dispatch_workflow(inputs):
    """Sends a workflow_dispatch event for demo-control.yml. Resource-changing: refuses unless CONFIRM is True."""
    if not CONFIRM:
        raise PermissionError("CONFIRM is False. Nothing was triggered.")
    url = "%s/repos/%s/actions/workflows/%s/dispatches" % (GITHUB_API, GITHUB_REPOSITORY, GITHUB_WORKFLOW_FILE)
    try:
        response = _github_request("POST", url, {"ref": GITHUB_REF, "inputs": inputs, "return_run_details": True})
    except Exception as error:
        return {"status": "Fail", "message": "The workflow wasn't triggered: " + redact(error)}
    if response.status_code in (200, 204):
        details = {}
        if response.status_code == 200:
            try:
                details = response.json() or {}
            except ValueError:
                details = {}
        return {"status": "Pass", "message": "demo-control.yml was triggered.",
                "github_run_id": details.get("workflow_run_id"), "github_run_url": details.get("html_url")}
    return {"status": "Fail", "message": ("GitHub returned HTTP %d. %s %s" % (
        response.status_code, GITHUB_HTTP_HINTS.get(response.status_code, ""), _error_text(response))).strip()}


TABLE_COLUMNS = {
    "validation_results": [
        ("run_id", "string"), ("source_notebook", "string"), ("check_id", "string"), ("area", "string"),
        ("target", "string"), ("status", "string"), ("message", "string"), ("evidence", "string"),
        ("remediation", "string"), ("doc_url", "string"), ("evaluation_only", "boolean"), ("timestamp_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    "scenario_runs": [
        ("request_id", "string"), ("run_id", "string"), ("source_notebook", "string"), ("requested_at_utc", "timestamp"),
        ("environment", "string"), ("target", "string"), ("profile", "string"), ("duration_minutes", "integer"),
        ("concurrency", "integer"), ("intensity", "integer"), ("reset_before", "boolean"), ("auto_cleanup", "boolean"),
        ("dispatch_mode", "string"), ("status", "string"), ("message", "string"), ("github_run_id", "long"),
        ("github_run_url", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    "telemetry_snapshots": [
        ("run_id", "string"), ("source_notebook", "string"), ("captured_at_utc", "timestamp"), ("analysis", "string"),
        ("target", "string"), ("resource_type", "string"), ("resource_name", "string"), ("instance_name", "string"),
        ("database_name", "string"), ("metric", "string"), ("bucket_utc", "timestamp"), ("value", "double"),
        ("unit", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("report_date_utc", "date"),
    ],
    "analysis_findings": [
        ("run_id", "string"), ("finding_id", "string"), ("source_notebook", "string"), ("captured_at_utc", "timestamp"),
        ("analysis", "string"), ("target", "string"), ("status", "string"), ("finding", "string"), ("evidence", "string"),
        ("request_id", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    # One row per incident window: a recorded incident request, or a detected signal episode (prepare_report_tables).
    "incident_windows": [
        ("incident_key", "string"), ("incident_label", "string"), ("incident_sort", "long"), ("source", "string"),
        ("target", "string"), ("scenario", "string"), ("signal_family", "string"), ("request_id", "string"),
        ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("duration_minutes", "double"),
        ("peak_value", "double"), ("peak_unit", "string"), ("correlation_status", "string"),
        ("correlation_evidence", "string"), ("run_id", "string"), ("captured_at_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
}


# The UTC calendar date of each row's primary time (the first one that isn't null). The Power BI history report
# (fabric/powerbi) relates every table to its Date table through this column.
REPORT_DATE_SOURCES = {
    "validation_results": ("timestamp_utc",),
    "scenario_runs": ("window_start_utc", "requested_at_utc"),
    "telemetry_snapshots": ("bucket_utc", "window_end_utc", "captured_at_utc"),
    "analysis_findings": ("window_start_utc", "captured_at_utc"),
    "incident_windows": ("window_start_utc",),
}


def _utc_date(value):
    """UTC calendar date of a timestamp; naive values are already UTC."""
    if value is None:
        return None
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        return None
    return (stamp.tz_convert("UTC") if stamp.tzinfo is not None else stamp).date()


def report_date(table, row):
    """report_date_utc for one row: the UTC date of the first primary time in REPORT_DATE_SOURCES that isn't null."""
    for name in REPORT_DATE_SOURCES.get(table, ()):
        value = _utc_date(row.get(name))
        if value is not None:
            return value
    return None


def _struct(table):
    from pyspark.sql import types as spark_types
    mapping = {
        "string": spark_types.StringType(), "boolean": spark_types.BooleanType(), "timestamp": spark_types.TimestampType(),
        "double": spark_types.DoubleType(), "long": spark_types.LongType(), "integer": spark_types.IntegerType(),
        "date": spark_types.DateType(),
    }
    return spark_types.StructType([spark_types.StructField(name, mapping[kind], True) for name, kind in TABLE_COLUMNS[table]])


def _spark_value(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.to_pydatetime(warn=False)
    if isinstance(value, float) and value != value:
        return None
    if hasattr(value, "item") and not isinstance(value, (str, bytes)):
        return value.item()
    return value


def _rows_frame(session, table, rows):
    """A Spark DataFrame with the TABLE_COLUMNS schema; report_date_utc comes from REPORT_DATE_SOURCES."""
    names = [name for name, _ in TABLE_COLUMNS[table]]
    data = [tuple(_spark_value(report_date(table, row) if name == "report_date_utc" else row.get(name)) for name in names)
            for row in rows]
    return session.createDataFrame(data, schema=_struct(table))


def write_delta(table, rows, replace_run=True):
    """Appends rows to a Delta table in the default lakehouse. Lakehouse data only: no Azure or Fabric resource
    changes. With replace_run, rows from this RUN_ID are deleted first, so re-running a cell doesn't duplicate them."""
    if not rows:
        print("%s: nothing to save." % table)
        return 0
    session = _spark()
    if session is None:
        print("%s: Spark isn't available, so nothing was saved." % table)
        return 0
    if not runtime_context().get("defaultLakehouseId"):
        print("%s: no default lakehouse is attached, so nothing was saved. Attach %s and run the cell again." % (table, LAKEHOUSE_NAME))
        return 0
    frame = _rows_frame(session, table, rows)
    if replace_run and session.catalog.tableExists(table):
        session.sql("DELETE FROM %s WHERE run_id = '%s'" % (table, RUN_ID))
    # One Parquet file per write keeps every table far below Direct Lake's limit of 1,000 files per table (F16).
    frame.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
    print("%s: saved %d row(s) for run %s." % (table, len(rows), RUN_ID))
    return len(rows)


def submit_simulation_request(request):
    """Triggers demo-control.yml (P9) or prints the manual steps, and records the request in scenario_runs.
    Resource-changing: refuses unless CONFIRM is True."""
    if not CONFIRM:
        raise PermissionError("CONFIRM is False. Nothing was triggered.")
    now = utc_now()
    previous = _DISPATCH_GUARD.get(request["fingerprint"])
    if previous and (now - previous).total_seconds() < 120:
        print("The same request was sent less than 2 minutes ago, so it wasn't sent again. Wait, or change a parameter.")
        return None
    window_end = now if request["profile"] == "off" else now + dt.timedelta(minutes=request["duration_minutes"])
    record = {
        "request_id": uuid.uuid4().hex, "run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "requested_at_utc": now,
        "environment": GITHUB_ENVIRONMENT, "target": request["target"], "profile": request["profile"],
        "duration_minutes": request["duration_minutes"], "concurrency": request["concurrency"], "intensity": request["intensity"],
        "reset_before": request["reset_before"], "auto_cleanup": request["auto_cleanup"],
        "github_run_id": None, "github_run_url": None, "window_start_utc": now, "window_end_utc": window_end,
    }
    if KEY_VAULT_URL:
        record["dispatch_mode"] = "workflow_dispatch"
        record.update(dispatch_workflow(request["inputs"]))
        if record["status"] != "Pass":
            print_manual_trigger(request)
    else:
        record.update(dispatch_mode="manual", status="Manual action required",
                      message="P9 isn't configured: run the workflow from the printed link or command. The window assumes it starts now.")
        print_manual_trigger(request)
    _DISPATCH_GUARD[request["fingerprint"]] = now
    SESSION_REQUESTS.append(record)
    print("%s: %s" % (record["status"], redact(record["message"])))
    if SAVE_RESULTS:
        write_delta("scenario_runs", [record], replace_run=False)
    return record


def scenario_windows(start, end):
    """Scenario windows that overlap [start, end]: this session's requests plus the scenario_runs table."""
    found = {record["request_id"]: record for record in SESSION_REQUESTS}
    session = _spark()
    if session is not None and runtime_context().get("defaultLakehouseId"):
        try:
            if session.catalog.tableExists("scenario_runs"):
                from pyspark.sql import functions as spark_functions
                rows = session.table("scenario_runs").select(
                    "request_id", "profile", "target", "status",
                    spark_functions.unix_timestamp("window_start_utc").alias("start_epoch"),
                    spark_functions.unix_timestamp("window_end_utc").alias("end_epoch")).collect()
                for row in rows:
                    if row["request_id"] not in found and row["start_epoch"] is not None and row["end_epoch"] is not None:
                        found[row["request_id"]] = {
                            "request_id": row["request_id"], "profile": row["profile"], "target": row["target"], "status": row["status"],
                            "window_start_utc": dt.datetime.fromtimestamp(row["start_epoch"], tz=dt.timezone.utc),
                            "window_end_utc": dt.datetime.fromtimestamp(row["end_epoch"], tz=dt.timezone.utc)}
        except Exception as error:
            print("scenario_runs couldn't be read: " + redact(error))
    windows = []
    for record in found.values():
        if record["window_end_utc"] < start or record["window_start_utc"] > end or record["status"] == "Fail":
            continue
        target = record["target"]
        windows.append({
            "request_id": record["request_id"], "profile": record["profile"], "status": record["status"],
            "targets": ["T1", "T2", "T3"] if target == "all" else [target],
            "start": max(record["window_start_utc"], start), "end": min(record["window_end_utc"], end),
            "window_start_utc": record["window_start_utc"], "window_end_utc": record["window_end_utc"],
            "label": "%s (%s)" % (record["profile"], target),
        })
    return sorted(windows, key=lambda item: item["start"])


# ---- Report tables for the Power BI history report (fabric/powerbi; Direct Lake on OneLake) --------------------

SPARK_DDL_TYPES = {"string": "STRING", "boolean": "BOOLEAN", "timestamp": "TIMESTAMP", "double": "DOUBLE", "long": "BIGINT",
                   "integer": "INT", "date": "DATE"}
# OPTIMIZE compacts a table above this many files, far below Direct Lake's limit of 1,000 files per table (F16).
OPTIMIZE_FILE_THRESHOLD = 200
_INCIDENT_KEY = re.compile(r"(request|detected):[A-Za-z0-9:._-]{1,120}")


def _as_utc(value):
    stamp = pd.Timestamp(value)
    return (stamp.tz_localize("UTC") if stamp.tzinfo is None else stamp.tz_convert("UTC")).to_pydatetime()


def _epoch_utc(value):
    return None if value is None else dt.datetime.fromtimestamp(value, tz=dt.timezone.utc)


def _json_object(text):
    """Parses stored evidence JSON back into a dict; anything else is kept as text."""
    try:
        value = json.loads(text or "{}")
    except ValueError:
        return {"raw": text}
    return value if isinstance(value, dict) else {"value": value}


def _sql_in_utc(session, statement):
    """Runs one SQL statement with the session time zone set to UTC, so to_date() returns the UTC calendar date."""
    previous = session.conf.get("spark.sql.session.timeZone")
    session.conf.set("spark.sql.session.timeZone", "UTC")
    try:
        return session.sql(statement)
    finally:
        session.conf.set("spark.sql.session.timeZone", previous)


def _incident_row(key, source, target, family, start, end, peak, scenario=None, request_id=None, status="Pass", evidence=None):
    """One incident_windows row. The label reads like "2026-10-21 14:05 UTC \u00b7 T1 \u00b7 Blocking (detected)", and
    incident_sort is the negative epoch minute of the start, so the report's picker lists the newest first."""
    start, end = _as_utc(start), _as_utc(end)
    origin = "detected" if source == "Detected signal" else "%s, requested" % scenario
    return {
        "incident_key": key,
        "incident_label": "%s UTC \u00b7 %s \u00b7 %s (%s)" % (start.strftime("%Y-%m-%d %H:%M"), target, SIGNAL_LABELS.get(family, "Activity"), origin),
        "incident_sort": -int(start.timestamp() // 60), "source": source, "target": target, "scenario": scenario,
        "signal_family": family, "request_id": request_id, "window_start_utc": start, "window_end_utc": end,
        "duration_minutes": round((end - start).total_seconds() / 60.0, 1), "peak_value": peak,
        "peak_unit": SIGNAL_UNITS.get(family, ""), "correlation_status": status,
        "correlation_evidence": redact(json.dumps(evidence or {}, default=str, sort_keys=True)),
        "run_id": RUN_ID, "captured_at_utc": utc_now(),
    }


def _series_peak(family, target, start, end):
    series = episode_series(family)
    if series is None or series.empty:
        return None
    part = series[(series["Target"] == target) & (series["Bucket"] >= start) & (series["Bucket"] < end)]
    return None if part.empty else round(_safe_number(part["Value"].max()), 3)


def incident_window_rows():
    """This run's incident windows, as rows (writes nothing). Recorded: each incident request in scenario_runs that
    wasn't a Fail, with its signal family, its peak, and section 8's correlation verdict. Detected: each episode that
    EPISODE_RULES find in the telemetry frames; one that overlaps a recorded window on the same target (within
    TELEMETRY_LAG_MINUTES) and matches one of its expected signals becomes evidence for that window instead of a row."""
    lag = dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES))
    recorded = []
    for window in scenario_windows(WINDOW_START, WINDOW_END):
        if not window["profile"].startswith("incident:"):
            continue
        family = expected_signals(window["profile"])[0]
        start, end = window["window_start_utc"], window["window_end_utc"]
        for target in window["targets"]:
            finding = next((item for item in FINDINGS if item["analysis"] == "correlation" and item["target"] == target
                            and item["request_id"] == window["request_id"]), None)
            if finding is not None:
                status, evidence = finding["status"], _json_object(finding["evidence"])
            else:
                status = "Not selected" if target not in SELECTED_TARGETS else "Manual action required"
                evidence = {"note": "No correlation verdict in this run: run section 8, Correlation."}
            key = "request:%s" % window["request_id"] + ("" if len(window["targets"]) == 1 else ":" + target)
            row = _incident_row(key, "Recorded request", target, family, start, end, _series_peak(family, target, start, end + lag),
                                scenario=window["profile"].split(":", 1)[1], request_id=window["request_id"], status=status)
            row["_evidence"], row["_profile"] = evidence, window["profile"]
            recorded.append(row)
    detected = []
    for family, threshold, strict, min_bins in EPISODE_RULES:
        for episode in signal_episodes(episode_series(family), threshold, BIN, min_bins, strict):
            summary = {"family": family, "start": episode["start"], "end": episode["end"], "peak": round(episode["peak"], 3),
                       "bins": episode["bins"]}
            explained = [row for row in recorded if row["target"] == episode["target"] and family in expected_signals(row["_profile"])
                         and episode["start"] <= row["window_end_utc"] + lag and episode["end"] >= row["window_start_utc"]]
            for row in explained:
                row["_evidence"].setdefault("detectedEpisodes", []).append(summary)
                if family == row["signal_family"] and (row["peak_value"] is None or episode["peak"] > row["peak_value"]):
                    row["peak_value"] = round(episode["peak"], 3)
            if explained:
                continue
            rule = {"threshold": threshold, "strictlyAbove": strict, "minBins": min_bins, "binMinutes": BIN}
            detected.append(_incident_row("detected:%s:%s:%s" % (episode["target"], family, episode["start"].strftime("%Y%m%dT%H%M")),
                                          "Detected signal", episode["target"], family, episode["start"], episode["end"],
                                          round(episode["peak"], 3), evidence=dict(summary, rule=rule)))
    for row in recorded:
        row["correlation_evidence"] = redact(json.dumps(row.pop("_evidence"), default=str, sort_keys=True))
        row.pop("_profile")
    return recorded, detected


def reconcile_recorded(recorded, stored):
    """Recorded requests already in incident_windows (stored: incident_key to a dict with correlation_status,
    correlation_evidence, and peak_value) keep a stored Pass verdict when a later run sees only part of the window,
    and keep the higher peak. Returns the rows to upsert."""
    for row in recorded:
        previous = stored.get(row["incident_key"])
        if previous is None:
            continue
        if previous["correlation_status"] == "Pass" and row["correlation_status"] != "Pass":
            row["correlation_status"], row["correlation_evidence"] = previous["correlation_status"], previous["correlation_evidence"]
        if previous["peak_value"] is not None and (row["peak_value"] is None or previous["peak_value"] > row["peak_value"]):
            row["peak_value"] = previous["peak_value"]
    return recorded


def union_episodes(stored, detected):
    """Unions this run's detected rows with stored detected episodes (dicts with key, target, family, start, end, and
    peak) of the same target and family that overlap or touch them, keeping the earliest stored key, so a re-run never
    duplicates an episode. Returns (rows to upsert, stored keys that another row absorbed)."""
    fresh = [{"kind": "new", "key": row["incident_key"], "target": row["target"], "family": row["signal_family"],
              "start": _as_utc(row["window_start_utc"]), "end": _as_utc(row["window_end_utc"]), "peak": row["peak_value"], "row": row}
             for row in detected]
    old = [dict(item, kind="stored", start=_as_utc(item["start"]), end=_as_utc(item["end"])) for item in stored]
    rows, absorbed = [], []
    for target, family in sorted({(item["target"], item["family"]) for item in fresh}):
        items = sorted((item for item in old + fresh if item["target"] == target and item["family"] == family),
                       key=lambda item: (item["start"], item["kind"] != "stored", item["key"]))
        clusters = []
        for item in items:
            if clusters and item["start"] <= clusters[-1]["end"]:
                clusters[-1]["end"] = max(clusters[-1]["end"], item["end"])
                clusters[-1]["items"].append(item)
            else:
                clusters.append({"start": item["start"], "end": item["end"], "items": [item]})
        for cluster in clusters:
            new_items = [item for item in cluster["items"] if item["kind"] == "new"]
            if not new_items:
                continue
            kept = [item for item in cluster["items"] if item["kind"] == "stored"]
            key = kept[0]["key"] if kept else new_items[0]["key"]
            absorbed += [item["key"] for item in kept[1:]]
            peaks = [item["peak"] for item in cluster["items"] if item["peak"] is not None]
            evidence = _json_object(new_items[0]["row"]["correlation_evidence"])
            if len(cluster["items"]) > 1:
                evidence["mergedFrom"] = sorted({item["key"] for item in cluster["items"]})
            rows.append(_incident_row(key, "Detected signal", target, family, cluster["start"], cluster["end"],
                                      max(peaks) if peaks else None, evidence=evidence))
    return rows, absorbed


def merge_incident_windows(session, recorded, detected):
    """Upserts this run's incident windows into incident_windows (MERGE on incident_key). Detected episodes are first
    unioned with the stored detected rows of the same target and family that overlap or touch them, keeping the
    earliest stored key, so a re-run never duplicates an episode. A recorded request keeps a stored Pass verdict
    when a later run sees only part of its window, and the higher peak. Lakehouse data only."""
    from pyspark.sql import functions as spark_functions
    table = session.table("incident_windows")
    upserts, absorbed = {}, []
    if recorded:
        stored = {row["incident_key"]: row.asDict() for row in table.where(spark_functions.col("incident_key").isin([row["incident_key"] for row in recorded]))
                  .select("incident_key", "correlation_status", "correlation_evidence", "peak_value").collect()}
        upserts.update((row["incident_key"], row) for row in reconcile_recorded(recorded, stored))
    if detected:
        step = dt.timedelta(minutes=int(BIN))
        nearby = table.where((spark_functions.col("source") == "Detected signal")
                             & (spark_functions.col("window_end_utc") >= spark_functions.lit(WINDOW_START - step))
                             & (spark_functions.col("window_start_utc") <= spark_functions.lit(WINDOW_END + step)))
        stored = [{"key": row["incident_key"], "target": row["target"], "family": row["signal_family"],
                   "start": _epoch_utc(row["start_epoch"]), "end": _epoch_utc(row["end_epoch"]), "peak": row["peak_value"]}
                  for row in nearby.select("incident_key", "target", "signal_family", "peak_value",
                                           spark_functions.unix_timestamp("window_start_utc").alias("start_epoch"),
                                           spark_functions.unix_timestamp("window_end_utc").alias("end_epoch")).collect()
                  if row["start_epoch"] is not None and row["end_epoch"] is not None]
        rows, absorbed = union_episodes(stored, detected)
        upserts.update((row["incident_key"], row) for row in rows)
    if upserts:
        _rows_frame(session, "incident_windows", list(upserts.values())).createOrReplaceTempView("incident_windows_updates")
        session.sql("MERGE INTO incident_windows AS target USING incident_windows_updates AS source "
                    "ON target.incident_key = source.incident_key WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    absorbed = sorted(key for key in set(absorbed) - set(upserts) if _INCIDENT_KEY.fullmatch(key))
    if absorbed:
        session.sql("DELETE FROM incident_windows WHERE incident_key IN (%s)" % ", ".join("'%s'" % key for key in absorbed))
    return len(upserts), len(absorbed)


def prepare_report_tables():
    """Readies the five tables that the Power BI history report reads (SM_DatabaseHub_History, Direct Lake on OneLake),
    which can't frame a missing table or column: CREATE TABLE IF NOT EXISTS for each, report_date_utc added to tables
    written before it existed and backfilled once, this run's incident windows merged, and OPTIMIZE for any table with
    more than OPTIMIZE_FILE_THRESHOLD files. Lakehouse data only: no Azure or Fabric resource changes."""
    session = _spark()
    if session is None or not runtime_context().get("defaultLakehouseId"):
        print("Report tables: Spark or the default lakehouse isn't available, so nothing was prepared. Attach %s." % LAKEHOUSE_NAME)
        return {}
    summary = {"created": [], "columnAdded": [], "backfilled": {}, "optimized": []}
    for table, columns in TABLE_COLUMNS.items():
        if not session.catalog.tableExists(table):
            summary["created"].append(table)
        session.sql("CREATE TABLE IF NOT EXISTS %s (%s) USING DELTA" % (table, ", ".join("%s %s" % (name, SPARK_DDL_TYPES[kind]) for name, kind in columns)))
        if "report_date_utc" not in {field.name for field in session.table(table).schema.fields}:
            session.sql("ALTER TABLE %s ADD COLUMNS (report_date_utc DATE)" % table)
            summary["columnAdded"].append(table)
        missing = int(session.sql("SELECT COUNT(*) AS n FROM %s WHERE report_date_utc IS NULL" % table).collect()[0]["n"])
        if missing:
            # Rows written before the column existed (also when an append with mergeSchema added it): a one-time backfill.
            _sql_in_utc(session, "UPDATE %s SET report_date_utc = to_date(coalesce(%s)) WHERE report_date_utc IS NULL"
                        % (table, ", ".join(REPORT_DATE_SOURCES[table])))
            summary["backfilled"][table] = missing
    recorded, detected = incident_window_rows()
    summary["incidentWindowsMerged"], summary["incidentWindowsAbsorbed"] = merge_incident_windows(session, recorded, detected)
    for table in TABLE_COLUMNS:
        files = session.sql("DESCRIBE DETAIL %s" % table).collect()[0]["numFiles"]
        if files and int(files) > OPTIMIZE_FILE_THRESHOLD:
            session.sql("OPTIMIZE %s" % table)
            summary["optimized"].append(table)
    print("Report tables ready for the Power BI history report: " + json.dumps(summary, sort_keys=True))
    return summary


WINDOW_START, WINDOW_END, BIN = analysis_window()
WINDOWS = []
print("Helpers ready. Run ID: %s. Targets: %s." % (RUN_ID, ", ".join(SELECTED_TARGETS)))

## 3. Simulation control

Pick a profile in the parameters cell (`PROFILE`, `SIM_TARGET`, `DURATION_MINUTES`, and the rest), then run the cells below in order:

1. **Plan** (always safe): validates the request and shows exactly what would be sent. No network calls.
2. **P9 readiness** (read-only): when `KEY_VAULT_URL` is set, checks that the GitHub token secret can be read. The value is never shown.
3. **Send** (resource-changing, needs `CONFIRM = True`): triggers `demo-control.yml`, or prints the link and command without P9. The request is recorded in `scenario_runs` either way.
4. **Run status** (read-only, needs P9): shows the GitHub Actions run that the request started.

**The `demo-control.yml` input contract** (ADR 0011). Values are strings, as `workflow_dispatch` expects:

| Workflow input | From parameter | Meaning |
|---|---|---|
| `environment` | `GITHUB_ENVIRONMENT` | GitHub environment: its variables, OIDC subject, and approvals |
| `target` | `SIM_TARGET` | `all`, `T1`, `T2`, or `T3`; an incident uses one target |
| `scenario` | `PROFILE` | `baseline`, `business-day`, `off`, or `incident:<scenario>` |
| `duration` | `DURATION_MINUTES` | Maximum minutes; the run also stops at auto-shutdown |
| `concurrency` | `CONCURRENCY` | Sessions; omitted when 0, so the scenario's default applies |
| `intensity` | `INTENSITY` | Relative load level; omitted when 0, so the scenario's default applies |
| `reset-before` | `RESET_BEFORE` | Reset `ContosoCare` to its seed state first |
| `auto-cleanup` | `AUTO_CLEANUP` | Stop and clean up at the end of the run |

Each scenario also declares its own maximum duration, concurrency, and intensity (ADR 0010), which the runner enforces, and the demo database rejects runs longer than 4 hours or with more than 64 sessions.

In [ ]:
# Simulation plan: validates the request and shows what would be sent. No network calls.
REQUEST = build_simulation_request(PROFILE, SIM_TARGET, DURATION_MINUTES, CONCURRENCY, INTENSITY, RESET_BEFORE, AUTO_CLEANUP)
print_simulation_plan(REQUEST)

In [ ]:
# P9 readiness: checks that the GitHub token secret can be read from Key Vault. Read-only; the value is never shown.
if not KEY_VAULT_URL:
    add_finding("simulation_control", "github", "Not selected",
                "P9 isn't configured, so this notebook prints the GitHub link and command instead of triggering the workflow.", "p9")
else:
    try:
        readable = bool(_notebookutils().credentials.getSecret(KEY_VAULT_URL, GITHUB_TOKEN_SECRET_NAME))
        if readable:
            add_finding("simulation_control", "github", "Pass", "The GitHub token secret is readable (its value isn't shown).", "p9")
        else:
            add_finding("simulation_control", "github", "Fail", "The GitHub token secret is empty.", "p9")
    except Exception as error:
        add_finding("simulation_control", "github", "Fail",
                    "The GitHub token secret couldn't be read: %s. You need permission to read secrets in the vault (an access "
                    "policy with Get, or the Key Vault Secrets User role), and the vault must allow your network path; tenant "
                    "policies can turn off its public network access." % redact(error), "p9")

> **Resource-changing cell.** With `CONFIRM = True`, the next cell triggers `demo-control.yml`, which changes the activity on the demo servers (and, with `reset-before`, resets `ContosoCare` to its seed state). With P9 it sends the request to GitHub; without P9 it prints the link and command for you to run, and records the request as **Manual action required**. With `CONFIRM = False` (the default) it only prints a dry run. Sending the same request again within 2 minutes is skipped, so an accidental re-run doesn't start a second run.

In [ ]:
# RESOURCE-CHANGING CELL: triggers demo-control.yml (GitHub Actions), which changes the activity on the demo servers.
if not CONFIRM:
    print("Dry run: CONFIRM is False, so nothing was triggered or recorded. Review the plan above, set CONFIRM = True in the "
          "parameters cell, and run the cells from there down to this one again.")
else:
    LAST_REQUEST = submit_simulation_request(REQUEST) or LAST_REQUEST

In [ ]:
# Run status of the last triggered request (read-only GitHub call; needs P9).
if not LAST_REQUEST or not LAST_REQUEST.get("github_run_id"):
    print("No GitHub Actions run to check in this session. Runs started by hand appear at %s/actions/workflows/%s" % (REPO_URL, GITHUB_WORKFLOW_FILE))
else:
    run_url = "%s/repos/%s/actions/runs/%d" % (GITHUB_API, GITHUB_REPOSITORY, int(LAST_REQUEST["github_run_id"]))
    try:
        response = _github_request("GET", run_url)
        if response.status_code == 200:
            run = response.json()
            print("Run %s: status %s, conclusion %s. %s" % (run.get("id"), run.get("status"), run.get("conclusion") or "pending", run.get("html_url")))
        else:
            print("GitHub returned HTTP %d. %s" % (response.status_code, GITHUB_HTTP_HINTS.get(response.status_code, "")))
    except Exception as error:
        print("The run status couldn't be read: " + redact(error))

## 4. Telemetry access

The analysis uses the documented telemetry REST endpoint (`POST https://adx.centralus.arcdataservices.com/kusto/v1/rest/query`, database `ArcSqlTelemetry`), the same data as Database Hub **Performance**. Its documented token audience is `https://kusto.kusto.windows.net`; a notebook can only ask for its `kusto` audience key, and **whether the endpoint accepts that token isn't documented**. The next cell runs the article's connectivity query once:

- **Pass:** the analysis below runs in this notebook.
- **Manual action required:** every analysis cell prints its KQL instead. Paste it into the Azure Data Explorer web UI (connection `https://adx.centralus.arcdataservices.com/kusto/`, database `ArcSqlTelemetry`), or create a Real-Time Dashboard over the same endpoint (section 10).
- **Fail:** HTTP 403. You need Reader on the subscription, and `Microsoft.Sql` must be registered on it.

It also sets the analysis window. Telemetry keeps up to 7 days and arrives 3–15 minutes after the activity.

In [ ]:
# Telemetry access: validates the undocumented notebook kusto-token path once, and sets the analysis window. Read-only.
WINDOW_START, WINDOW_END, BIN = analysis_window()
print("Analysis window: %s to %s UTC, %d-minute bins." % (WINDOW_START.strftime("%Y-%m-%d %H:%M"), WINDOW_END.strftime("%Y-%m-%d %H:%M"), BIN))

PROBE_KQL = """SqlServerCPUUtilization
| where SampleTimeUTC > ago(15m)
| summarize Resources = dcount(ResourceID) by ResourceType"""
probe_status, probe_frame, probe_message = _post_kql(PROBE_KQL)
TELEMETRY_STATE.update(checked=True, available=(probe_status == 200), message=probe_message)
if probe_status == 200:
    add_finding("telemetry_access", "fabric", "Pass",
                "The telemetry endpoint accepted the notebook's kusto token, so the analysis runs here.", "token")
elif probe_status == 403:
    add_finding("telemetry_access", "fabric", "Fail",
                probe_message + " Ask for Reader on the subscription (P8) and register Microsoft.Sql (P2).", "token")
else:
    add_finding("telemetry_access", "fabric", "Manual action required",
                "The notebook can't query the telemetry endpoint directly (%s). Each analysis cell prints its KQL for the Azure Data Explorer web UI." % probe_message,
                "token", evidence={"httpStatus": probe_status})

## 5. Estate pulse

Which demo resources are reporting, how many telemetry tables each sends, and how old the last sample is (the telemetry article's query 1, scoped to `RESOURCE_GROUP` and the demo's names). A resource that's missing either isn't running, hasn't reported yet, or doesn't have monitoring enabled: an empty result never means healthy.

In [ ]:
# Estate pulse: resources reporting and the age of their last sample. Read-only.
PULSE_KQL = kql_prelude(WINDOW_START, WINDOW_END, BIN) + """
union withsource=TableName SqlServer*
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Rows = count(), Tables = dcount(TableName), LastSampleUTC = max(SampleTimeUTC), Instances = dcount(SqlServerInstanceName)
        by Target, ResourceType, ResourceName
| order by Target asc"""

pulse = query_telemetry(PULSE_KQL, "Estate pulse")
FRAMES["pulse"] = pulse
if pulse is not None:
    if not pulse.empty:
        pulse["LastSampleAgeMinutes"] = ((WINDOW_END - pulse["LastSampleUTC"]).dt.total_seconds() / 60).round(1)
        display_frame(pulse[["Target", "ResourceType", "ResourceName", "Tables", "Rows", "Instances", "LastSampleAgeMinutes"]])
    for target in SELECTED_TARGETS:
        rows = pulse[pulse["Target"] == target] if not pulse.empty else pulse
        if rows.empty:
            add_finding("estate_pulse", target, "Warning",
                        "%s sent no telemetry in the window: not running, not reporting yet (3-15 minutes), or monitoring isn't enabled." % target, "reporting")
        else:
            age = float(rows["LastSampleAgeMinutes"].min())
            status = "Pass" if age <= 15 else "Warning"
            add_finding("estate_pulse", target, status,
                        "%s (%s) reports %d table(s); the last sample is %.1f minutes old." % (target, rows["ResourceName"].iloc[0], int(rows["Tables"].max()), age),
                        "reporting", evidence={"tables": int(rows["Tables"].max()), "rows": int(rows["Rows"].sum()), "lastSampleAgeMinutes": age})
    add_snapshot("estate_pulse", pulse, WINDOW_START, WINDOW_END, metrics=["Rows", "Tables", "LastSampleAgeMinutes"],
                 units={"Rows": "rows", "Tables": "tables", "LastSampleAgeMinutes": "minutes"})

## 6. Locks and blocking

| Signal | Source | How it's computed |
|---|---|---|
| `Processes blocked` timeline | `SqlServerPerformanceCountersCommon`, `CounterType` 65792 (gauge) | Maximum per bin |
| Lock-category waits | `SqlServerWaitStats` (cumulative) | Per-series deltas, resets skipped, summed per bin |
| `Active Transactions` | Performance counters (gauge, per database) | Maximum across databases per bin |
| Active sessions in `ContosoCare` | `SqlServerActiveSessions` (gauge) | Distinct sessions per sample, maximum per bin |
| Deadlocks | `Number of Deadlocks/sec`, `CounterType` 272696576 (cumulative) | Per-series deltas per bin (`_Total` instance when present) |
| **Incidents detected** | `Processes blocked` | Consecutive bins with at least one blocked process |

An incident inside a recorded scenario window for the same target is expected (**Pass**); one outside any window is unexpected (**Warning**). The telemetry describes resources, not queries: for the blocking chain itself, use SSMS or the `diagnose-blocking-and-deadlocks` skill on the target.

In [ ]:
# Locks and blocking: gauges, cumulative counters and waits as per-bin deltas, sessions, incidents, and a chart. Read-only.
prelude = kql_prelude(WINDOW_START, WINDOW_END, BIN)

GAUGES_KQL = prelude + """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 65792
| where CounterName in ('Processes blocked', 'Active Transactions', 'User Connections')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize MaxValue = max(CounterValue), AvgValue = avg(CounterValue)
        by Target, ResourceName, SqlServerInstanceName, CounterName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc
| take 50000"""

# Cumulative counters: keep the first and last sample per series and bin, then sum increases between adjacent
# samples and skip any interval where the value went down (a restart or failover reset).
RATES_KQL = prelude + """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 272696576
| where CounterName in ('Number of Deadlocks/sec', 'Batch Requests/sec', 'Transactions/sec', 'Log Bytes Flushed/sec')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstV) = arg_min(SampleTimeUTC, CounterValue), (LastT, LastV) = arg_max(SampleTimeUTC, CounterValue)
        by Target, ResourceID, SqlServerInstanceName, CounterName, InstanceName, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), V = pack_array(FirstV, LastV)
| mv-expand T to typeof(datetime), V to typeof(real)
| distinct Target, ResourceID, SqlServerInstanceName, CounterName, InstanceName, Bucket, T, V
| sort by ResourceID asc, SqlServerInstanceName asc, CounterName asc, InstanceName asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and CounterName == prev(CounterName) and InstanceName == prev(InstanceName) and V >= prev(V)
| extend Delta = iff(Keep, V - prev(V), real(0)), Seconds = iff(Keep, datetime_diff('second', T, prev(T)), long(0))
| summarize Delta = sum(Delta), Seconds = sum(Seconds) by Target, CounterName, InstanceName, Bucket
| order by Bucket asc
| take 50000"""

WAITS_KQL = prelude + """
SqlServerWaitStats
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where WaitCategory !in ('Idle', 'Unknown')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstW, FirstC) = arg_min(SampleTimeUTC, WaitTimeMs, WaitingTasksCount),
            (LastT, LastW, LastC) = arg_max(SampleTimeUTC, WaitTimeMs, WaitingTasksCount)
        by Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), W = pack_array(FirstW, LastW), C = pack_array(FirstC, LastC)
| mv-expand T to typeof(datetime), W to typeof(long), C to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket, T, W, C
| sort by ResourceID asc, SqlServerInstanceName asc, WaitType asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and WaitType == prev(WaitType) and W >= prev(W) and C >= prev(C)
| extend WaitMs = iff(Keep, W - prev(W), long(0)), Tasks = iff(Keep, C - prev(C), long(0))
| summarize WaitMs = sum(WaitMs), Tasks = sum(Tasks) by Target, WaitCategory, PageIoLatch = WaitType startswith 'PAGEIOLATCH', Bucket
| where WaitMs > 0
| order by Bucket asc
| take 50000"""

SESSIONS_KQL = prelude + """
SqlServerActiveSessions
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where DatabaseName =~ _db
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Sessions = dcount(SessionID), Running = dcountif(SessionID, SessionStatus == 'running')
        by Target, SqlServerInstanceName, SampleTimeUTC
| summarize MaxSessions = max(Sessions), AvgSessions = avg(Sessions), MaxRunning = max(Running)
        by Target, SqlServerInstanceName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc"""


def prefer_total(rates):
    """Per target, counter, and bin: the _Total instance when the counter has one, otherwise the sum of instances."""
    if rates is None or rates.empty:
        return rates
    frame = rates.copy()
    frame["IsTotal"] = frame["InstanceName"].astype(str).str.strip().str.lower() == "_total"
    has_total = frame.groupby(["Target", "CounterName"])["IsTotal"].transform("any")
    chosen = frame[(~has_total) | frame["IsTotal"]]
    return chosen.groupby(["Target", "CounterName", "Bucket"], as_index=False)[["Delta", "Seconds"]].sum()


def blocking_incidents(gauges, bin_minutes):
    """Groups consecutive bins with at least one blocked process into incidents, per target (signal_episodes)."""
    blocked = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
    return signal_episodes(blocked, 0, bin_minutes, 1, strict=True)


gauges = query_telemetry(GAUGES_KQL, "Blocking and transaction gauges")
rates = query_telemetry(RATES_KQL, "Cumulative counters as deltas")
waits = query_telemetry(WAITS_KQL, "Wait deltas by category")
sessions = query_telemetry(SESSIONS_KQL, "Active sessions in the demo database")
FRAMES.update(gauges=gauges, rates=prefer_total(rates), waits=waits, sessions=sessions)
WINDOWS = scenario_windows(WINDOW_START, WINDOW_END)

blocked_series = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
lock_series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == "Lock")
if lock_series is not None:
    lock_series["Value"] = lock_series["Value"] / 1000.0
transactions_series = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Active Transactions")
sessions_series = to_series(sessions, "MaxSessions", "sum")
deadlock_series = to_series(FRAMES["rates"], "Delta", "sum", lambda f: f["CounterName"] == "Number of Deadlocks/sec")
plot_panels("Locks and blocking", [
    ("Processes blocked (maximum per bin)", blocked_series, "processes"),
    ("Lock waits (seconds per bin)", lock_series, "seconds"),
    ("Active transactions (maximum across databases)", transactions_series, "transactions"),
    ("Active sessions in %s (maximum per bin)" % DEMO_DATABASE, sessions_series, "sessions"),
    ("Deadlocks (per bin)", deadlock_series, "deadlocks"),
], WINDOWS)

if gauges is not None:
    incidents = blocking_incidents(gauges, BIN)
    if not incidents:
        print("No blocking incidents in the window.")
    for incident in incidents:
        expected = any(incident["target"] in window["targets"] and window["start"] <= incident["end"]
                       and incident["start"] <= window["end"] + dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES)) for window in WINDOWS)
        lock_seconds = 0.0
        if waits is not None and not waits.empty:
            inside = waits[(waits["Target"] == incident["target"]) & (waits["WaitCategory"] == "Lock")
                           & (waits["Bucket"] >= incident["start"]) & (waits["Bucket"] < incident["end"])]
            lock_seconds = round(_safe_number(inside["WaitMs"].sum()) / 1000.0, 1)
        add_finding("blocking", incident["target"], "Pass" if expected else "Warning",
                    "Blocking on %s from %s to %s UTC: up to %d blocked process(es), %.1f s of lock waits%s." % (
                        incident["target"], incident["start"].strftime("%H:%M"), incident["end"].strftime("%H:%M"),
                        int(incident["peak"]), lock_seconds, " during a recorded scenario" if expected else ", outside any recorded scenario"),
                    incident["start"].strftime("%Y%m%dT%H%M"), evidence={"peakBlocked": incident["peak"], "lockWaitSeconds": lock_seconds},
                    start=incident["start"].to_pydatetime(), end=incident["end"].to_pydatetime())

if deadlock_series is not None:
    for target, group in deadlock_series.groupby("Target"):
        total = int(round(_safe_number(group["Value"].sum())))
        if total > 0:
            add_finding("blocking", target, "Warning", "%d deadlock(s) on %s in the window." % (total, target), "deadlocks",
                        evidence={"deadlocks": total}, start=WINDOW_START, end=WINDOW_END)

add_snapshot("gauges", gauges, WINDOW_START, WINDOW_END, metric_column="CounterName", value_column="MaxValue", units={})
add_snapshot("counter_deltas", FRAMES["rates"], WINDOW_START, WINDOW_END, metric_column="CounterName", value_column="Delta")
add_snapshot("sessions", sessions, WINDOW_START, WINDOW_END, metrics=["MaxSessions", "AvgSessions", "MaxRunning"],
             units={"MaxSessions": "sessions", "AvgSessions": "sessions", "MaxRunning": "sessions"})

## 7. Overall consumption

| Signal | Source | Notes |
|---|---|---|
| CPU average, 95th percentile, maximum, and time above 80% | `SqlServerCPUUtilization` (gauge) | T1 and T2 read `SQLProcessCPUPercent`, the SQL Server process's share of the machine, because `AvgCPUPercent` stays at 0 for SQL Server in this preview (Phase 25a, 2026-10-23); both columns are in the table's documented schema. T3 reads `AvgCPUPercent`, a percentage of the service tier's limit |
| CPU timeline | `SqlServerCPUUtilization` | Average and maximum per bin |
| Wait profile | `SqlServerWaitStats` deltas | Share of wait time per category, `Idle` and `Unknown` excluded |
| `PAGEIOLATCH` waits | Buffer IO wait types | Data pages read from storage: scans, IO pressure, or a missing index |
| Storage IO | `SqlServerStorageIO` (cumulative) | Per-file deltas: MB read and written per bin; read and write latency for T1 and T2 only, because per-file latency isn't supported for Azure SQL Database |
| Memory clerks | `SqlServerMemoryUtilization` (gauge) | Largest clerk types at the latest sample |
| Database growth | `SqlServerDatabaseStorageUtilization` (gauge) | Used data and log space at the start and end of the window |
| **T3 `Log Rate Governor` waits** | `SqlServerWaitStats` | Azure SQL Database's log-rate governance; the T3 log-rate scenario's signature |
| Buffer Latch, Memory, Parallelism, Tran Log IO, and Network IO waits | `SqlServerWaitStats` deltas | The Phase 25b scenarios' signatures: tempdb contention (`PAGELATCH_*`), memory grants (`RESOURCE_SEMAPHORE`), parallel plans (`CXPACKET`, `CXCONSUMER`), commits (`WRITELOG`), and a slow client (`ASYNC_NETWORK_IO`); charted when present, and a finding when one passes `EPISODE_WAIT_SECONDS` and 10% of a target's wait time |

In [ ]:
# Overall consumption: CPU, waits, storage IO, memory, database growth, and T3 log-rate governance. Read-only.
CPU_SUMMARY_KQL = prelude + """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = round(avg(CpuPercent), 1), P95CPU = round(percentile(CpuPercent, 95), 1),
            MaxCPU = round(max(CpuPercent), 1), PctTimeAbove80 = round(100.0 * countif(CpuPercent > 80.0) / count(), 1)
        by Target, ResourceType, ResourceName, SqlServerInstanceName
| order by Target asc"""

CPU_TIMELINE_KQL = prelude + """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = avg(CpuPercent), MaxCPU = max(CpuPercent) by Target, ResourceName, SqlServerInstanceName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc"""

STORAGE_IO_KQL = prelude + """
SqlServerStorageIO
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where FileType in ('Data', 'Log')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, R1, W1, RS1, WS1, NR1, NW1) = arg_min(SampleTimeUTC, NumOfBytesRead, NumOfBytesWritten, IOStallReadMs, IOStallWriteMs, NumOfReads, NumOfWrites),
            (LastT, R2, W2, RS2, WS2, NR2, NW2) = arg_max(SampleTimeUTC, NumOfBytesRead, NumOfBytesWritten, IOStallReadMs, IOStallWriteMs, NumOfReads, NumOfWrites)
        by Target, ResourceID, SqlServerInstanceName, DatabaseID, FileID, FileType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), R = pack_array(R1, R2), W = pack_array(W1, W2), RS = pack_array(RS1, RS2),
         WS = pack_array(WS1, WS2), NR = pack_array(NR1, NR2), NW = pack_array(NW1, NW2)
| mv-expand T to typeof(datetime), R to typeof(long), W to typeof(long), RS to typeof(long), WS to typeof(long), NR to typeof(long), NW to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, DatabaseID, FileID, FileType, Bucket, T, R, W, RS, WS, NR, NW
| sort by ResourceID asc, SqlServerInstanceName asc, DatabaseID asc, FileID asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and DatabaseID == prev(DatabaseID) and FileID == prev(FileID)
                and R >= prev(R) and W >= prev(W) and RS >= prev(RS) and WS >= prev(WS) and NR >= prev(NR) and NW >= prev(NW)
| extend dR = iff(Keep, R - prev(R), long(0)), dW = iff(Keep, W - prev(W), long(0)), dRS = iff(Keep, RS - prev(RS), long(0)),
         dWS = iff(Keep, WS - prev(WS), long(0)), dNR = iff(Keep, NR - prev(NR), long(0)), dNW = iff(Keep, NW - prev(NW), long(0))
| summarize ReadMB = round(sum(dR) / 1048576.0, 2), WriteMB = round(sum(dW) / 1048576.0, 2), ReadStallMs = sum(dRS),
            WriteStallMs = sum(dWS), Reads = sum(dNR), Writes = sum(dNW) by Target, FileType, Bucket
| order by Bucket asc
| take 50000"""

MEMORY_KQL = prelude + """
SqlServerMemoryUtilization
| where SampleTimeUTC between ((_end - 1h) .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize arg_max(SampleTimeUTC, MemorySizeMB) by Target, SqlServerInstanceName, MemoryClerkType, MemoryClerkName
| summarize MemoryMB = round(sum(MemorySizeMB), 1), LastSampleUTC = max(SampleTimeUTC) by Target, SqlServerInstanceName, MemoryClerkType
| where MemoryMB >= 1
| order by Target asc, MemoryMB desc
| take 1000"""

GROWTH_KQL = prelude + """
SqlServerDatabaseStorageUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where DatabaseName =~ _db
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstUTC, StartUsedMB) = arg_min(SampleTimeUTC, DataSizeUsedMB),
            (LastUTC, EndUsedMB, AllocatedMB, LogUsedMB, VersionStoreMB) =
                arg_max(SampleTimeUTC, DataSizeUsedMB, DataSizeAllocatedMB, LogSizeUsedMB, PersistentVersionStoreSizeMB)
        by Target, ResourceName, SqlServerInstanceName, DatabaseName
| extend GrowthMB = round(EndUsedMB - StartUsedMB, 1)
| order by Target asc"""

cpu_summary = query_telemetry(CPU_SUMMARY_KQL, "CPU summary")
cpu = query_telemetry(CPU_TIMELINE_KQL, "CPU timeline")
storage_io = query_telemetry(STORAGE_IO_KQL, "Storage IO deltas")
memory = query_telemetry(MEMORY_KQL, "Memory clerks")
growth = query_telemetry(GROWTH_KQL, "Database growth")
FRAMES.update(cpu=cpu, storage_io=storage_io, memory=memory, growth=growth)

if cpu_summary is not None and not cpu_summary.empty:
    print("CPU over the window (T3: percentage of the service tier's limit):")
    display_frame(cpu_summary[["Target", "ResourceName", "SqlServerInstanceName", "AvgCPU", "P95CPU", "MaxCPU", "PctTimeAbove80"]])
    for _, row in cpu_summary.iterrows():
        hot = _safe_number(row["P95CPU"]) >= 80
        add_finding("consumption", row["Target"], "Warning" if hot else "Pass",
                    "%s CPU: average %.1f%%, 95th percentile %.1f%%, maximum %.1f%%, %.1f%% of the window above 80%%%s." % (
                        row["Target"], _safe_number(row["AvgCPU"]), _safe_number(row["P95CPU"]), _safe_number(row["MaxCPU"]),
                        _safe_number(row["PctTimeAbove80"]), ": sustained CPU pressure" if hot else ""),
                    "cpu", evidence={"avg": row["AvgCPU"], "p95": row["P95CPU"], "max": row["MaxCPU"]}, start=WINDOW_START, end=WINDOW_END)

waits = FRAMES.get("waits")
if waits is not None and not waits.empty:
    profile = waits.groupby(["Target", "WaitCategory"], as_index=False)["WaitMs"].sum()
    profile["WaitSeconds"] = (profile["WaitMs"] / 1000.0).round(1)
    profile["SharePct"] = (100.0 * profile["WaitMs"] / profile.groupby("Target")["WaitMs"].transform("sum")).round(1)
    print("Wait profile per target (share of non-idle wait time in the window):")
    display_frame(profile.sort_values(["Target", "SharePct"], ascending=[True, False])[["Target", "WaitCategory", "WaitSeconds", "SharePct"]])
    log_rate = profile[profile["WaitCategory"] == "Log Rate Governor"]
    for _, row in log_rate.iterrows():
        add_finding("consumption", row["Target"], "Warning",
                    "%s waited %.1f s on log-rate governance (%.1f%% of its wait time)." % (row["Target"], row["WaitSeconds"], row["SharePct"]),
                    "log_rate", evidence={"waitSeconds": row["WaitSeconds"], "sharePct": row["SharePct"]}, start=WINDOW_START, end=WINDOW_END)
    # The Phase 25b wait families: a finding when one passes EPISODE_WAIT_SECONDS and 10% of a target's wait time.
    for family in ("latch", "memory", "parallelism", "tran_log", "network"):
        part = profile[(profile["WaitCategory"] == WAIT_FAMILY_CATEGORIES[family]) & (profile["WaitSeconds"] >= EPISODE_WAIT_SECONDS)
                       & (profile["SharePct"] >= 10)]
        for _, row in part.iterrows():
            add_finding("consumption", row["Target"], "Warning",
                        "%s waited %.1f s on %s (%.1f%% of its wait time): %s." % (
                            row["Target"], row["WaitSeconds"], row["WaitCategory"], row["SharePct"], SIGNAL_LABELS[family].lower()),
                        family, evidence={"waitSeconds": row["WaitSeconds"], "sharePct": row["SharePct"]}, start=WINDOW_START, end=WINDOW_END)
    add_snapshot("wait_profile", profile.assign(Bucket=None), WINDOW_START, WINDOW_END, metric_column="WaitCategory", value_column="WaitSeconds")
    add_snapshot("wait_deltas", waits.assign(Metric=waits["WaitCategory"] + waits["PageIoLatch"].map({True: " (PAGEIOLATCH)", False: ""}).fillna("")),
                 WINDOW_START, WINDOW_END, metric_column="Metric", value_column="WaitMs")

if storage_io is not None and not storage_io.empty:
    io_view = storage_io.groupby("Target", as_index=False)[["ReadMB", "WriteMB", "ReadStallMs", "WriteStallMs", "Reads", "Writes"]].sum()
    io_view["ReadLatencyMs"] = (io_view["ReadStallMs"] / io_view["Reads"].where(io_view["Reads"] > 0)).round(2)
    io_view["WriteLatencyMs"] = (io_view["WriteStallMs"] / io_view["Writes"].where(io_view["Writes"] > 0)).round(2)
    # Per-file IO latency from this table isn't supported for Azure SQL Database (telemetry article, rule 5).
    io_view.loc[io_view["Target"] == "T3", ["ReadLatencyMs", "WriteLatencyMs"]] = None
    print("Storage IO in the window (data and log files):")
    display_frame(io_view[["Target", "ReadMB", "WriteMB", "Reads", "Writes", "ReadLatencyMs", "WriteLatencyMs"]])
    add_snapshot("storage_io", storage_io, WINDOW_START, WINDOW_END, metrics=["ReadMB", "WriteMB", "Reads", "Writes"],
                 units={"ReadMB": "MB", "WriteMB": "MB", "Reads": "reads", "Writes": "writes"})

if memory is not None and not memory.empty:
    top_memory = memory.sort_values("MemoryMB", ascending=False).groupby("Target").head(8)
    print("Largest memory clerk types at the latest sample:")
    display_frame(top_memory[["Target", "SqlServerInstanceName", "MemoryClerkType", "MemoryMB"]])
    add_snapshot("memory_clerks", top_memory.assign(Bucket=top_memory["LastSampleUTC"]), WINDOW_START, WINDOW_END,
                 metric_column="MemoryClerkType", value_column="MemoryMB")

if growth is not None and not growth.empty:
    print("Database %s storage over the window:" % DEMO_DATABASE)
    display_frame(growth[["Target", "ResourceName", "StartUsedMB", "EndUsedMB", "GrowthMB", "AllocatedMB", "LogUsedMB", "VersionStoreMB"]])
    add_snapshot("database_growth", growth.assign(Bucket=growth["LastUTC"]), WINDOW_START, WINDOW_END,
                 metrics=["StartUsedMB", "EndUsedMB", "GrowthMB", "LogUsedMB", "VersionStoreMB"],
                 units={"StartUsedMB": "MB", "EndUsedMB": "MB", "GrowthMB": "MB", "LogUsedMB": "MB", "VersionStoreMB": "MB"})

cpu_series = to_series(cpu, "AvgCPU", "max")
pageio_series = to_series(waits, "WaitMs", "sum", lambda f: f["PageIoLatch"].astype(bool))
if pageio_series is not None:
    pageio_series["Value"] = pageio_series["Value"] / 1000.0
log_rate_series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == "Log Rate Governor")
if log_rate_series is not None:
    log_rate_series["Value"] = log_rate_series["Value"] / 1000.0
family_series = {}
for family in ("latch", "memory", "parallelism", "tran_log", "network"):
    family_series[family] = episode_series(family)
io_series = None
if storage_io is not None and not storage_io.empty:
    io_series = to_series(storage_io.assign(TotalMB=storage_io["ReadMB"] + storage_io["WriteMB"]), "TotalMB", "sum")
plot_panels("Overall consumption", [
    ("CPU, average per bin (T3: % of the tier limit)", cpu_series, "% CPU"),
    ("PAGEIOLATCH (Buffer IO) waits, seconds per bin", pageio_series, "seconds"),
    ("Log Rate Governor waits, seconds per bin", log_rate_series, "seconds"),
    ("Data and log IO, MB read and written per bin", io_series, "MB"),
    ("Buffer Latch waits (PAGELATCH, tempdb contention), seconds per bin", family_series["latch"], "seconds"),
    ("Memory waits (RESOURCE_SEMAPHORE), seconds per bin", family_series["memory"], "seconds"),
    ("Parallelism waits (CXPACKET, CXCONSUMER), seconds per bin", family_series["parallelism"], "seconds"),
    ("Tran Log IO waits (WRITELOG), seconds per bin", family_series["tran_log"], "seconds"),
    ("Network IO waits (ASYNC_NETWORK_IO), seconds per bin", family_series["network"], "seconds"),
], WINDOWS)
add_snapshot("cpu", cpu, WINDOW_START, WINDOW_END, metrics=["AvgCPU", "MaxCPU"], units={"AvgCPU": "percent", "MaxCPU": "percent"})

## 8. Correlation: scenario windows and telemetry

Each recorded scenario window (from `scenario_runs`, including requests sent by hand from the printed link) is compared with the signals its profile should produce on its targets, from the window's start to `TELEMETRY_LAG_MINUTES` after its end:

| Profile | Expected signal |
|---|---|
| `incident:locking-blocking` | `Processes blocked` above 0 or Lock waits |
| `incident:cpu-pressure` | Average CPU in the window at least 1.5 times, and 10 points above, the same target's CPU outside incident windows |
| `incident:io-pressure` | `PAGEIOLATCH` waits, or read volume above the same target's level outside incident windows |
| `incident:inefficient-query` | Higher CPU, or `Log Rate Governor` waits (T3) |
| `incident:missing-index` | Higher CPU or `PAGEIOLATCH` waits |
| `incident:resource-governance` | Higher CPU (T3: at the service tier's limit) |
| `incident:log-rate-pressure` | `Log Rate Governor` waits (T3) |
| `incident:blocking-chain` | `Processes blocked` above 0 or Lock waits |
| `incident:deadlock-storm` | `Number of Deadlocks/sec` above 0, or blocking |
| `incident:tempdb-contention` | Buffer Latch waits (`PAGELATCH_*`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:memory-grant-pressure` | Memory waits (`RESOURCE_SEMAPHORE`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:parallel-scans` | Parallelism waits averaging at least `EPISODE_WAIT_SECONDS` per bin, or higher CPU |
| `incident:log-flush-waits` | Tran Log IO waits (`WRITELOG`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:slow-consumer` | Network IO waits (`ASYNC_NETWORK_IO`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:capacity-max` | Higher CPU or `PAGEIOLATCH` waits |
| `baseline`, `business-day` | Activity: batch requests or sessions in `ContosoCare` |

**Pass** means the signal appeared; **Warning** means it didn't (yet): telemetry lags 3–15 minutes, the run may not have started, or it ran on another target. **Manual action required** means the telemetry couldn't be queried here. These thresholds are demo heuristics for the presenter, not product behavior.

In [ ]:
# Correlation: did each recorded scenario window produce its expected signal? Read-only.
LAG = dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES))


def _slice(name, target, start, end):
    frame = FRAMES.get(name)
    if frame is None:
        return None
    if frame.empty:
        return frame
    part = frame[frame["Target"] == target]
    return part[(part["Bucket"] >= start) & (part["Bucket"] <= end)]


def _outside(name, target, incident_windows):
    frame = FRAMES.get(name)
    if frame is None or frame.empty:
        return frame
    part = frame[frame["Target"] == target]
    mask = pd.Series(True, index=part.index)
    for window in incident_windows:
        mask &= ~((part["Bucket"] >= window["start"]) & (part["Bucket"] <= window["end"] + LAG))
    return part[mask]


def evaluate_signal(signal, target, start, end, incident_windows):
    """Returns (observed, evidence): True or False when the data can tell, None when the telemetry wasn't available."""
    if signal == "blocking":
        gauges_part, waits_part = _slice("gauges", target, start, end), _slice("waits", target, start, end)
        if gauges_part is None and waits_part is None:
            return None, {}
        blocked = 0.0 if gauges_part is None or gauges_part.empty else _safe_number(
            gauges_part.loc[gauges_part["CounterName"] == "Processes blocked", "MaxValue"].max())
        lock_seconds = 0.0 if waits_part is None or waits_part.empty else _safe_number(
            waits_part.loc[waits_part["WaitCategory"] == "Lock", "WaitMs"].sum()) / 1000.0
        return blocked > 0 or lock_seconds > 0, {"maxProcessesBlocked": blocked, "lockWaitSeconds": round(lock_seconds, 1)}
    if signal == "cpu":
        inside = _slice("cpu", target, start, end)
        if inside is None:
            return None, {}
        if inside.empty:
            return False, {"cpuSamples": 0}
        inside_avg = _safe_number(inside["AvgCPU"].mean())
        outside = _outside("cpu", target, incident_windows)
        baseline = None if outside is None or outside.empty else _safe_number(outside["AvgCPU"].mean())
        observed = inside_avg >= 50 if baseline is None else inside_avg >= max(baseline * 1.5, baseline + 10)
        return observed, {"cpuAvgInWindow": round(inside_avg, 1), "cpuAvgOutside": None if baseline is None else round(baseline, 1)}
    if signal == "io":
        waits_part, io_part = _slice("waits", target, start, end), _slice("storage_io", target, start, end)
        if waits_part is None and io_part is None:
            return None, {}
        pageio_seconds = 0.0 if waits_part is None or waits_part.empty else _safe_number(
            waits_part.loc[waits_part["PageIoLatch"].astype(bool), "WaitMs"].sum()) / 1000.0
        read_rate, baseline_rate = 0.0, None
        if io_part is not None and not io_part.empty:
            read_rate = _safe_number(io_part.groupby("Bucket")["ReadMB"].sum().mean())
            outside = _outside("storage_io", target, incident_windows)
            if outside is not None and not outside.empty:
                baseline_rate = _safe_number(outside.groupby("Bucket")["ReadMB"].sum().mean())
        more_reads = read_rate > 0 and (baseline_rate is None or read_rate >= max(baseline_rate * 1.5, baseline_rate + 1))
        return pageio_seconds > 0 or more_reads, {"pageIoLatchSeconds": round(pageio_seconds, 1), "readMBPerBin": round(read_rate, 2),
                                                  "readMBPerBinOutside": None if baseline_rate is None else round(baseline_rate, 2)}
    if signal == "log_rate":
        waits_part = _slice("waits", target, start, end)
        if waits_part is None:
            return None, {}
        seconds = 0.0 if waits_part.empty else _safe_number(waits_part.loc[waits_part["WaitCategory"] == "Log Rate Governor", "WaitMs"].sum()) / 1000.0
        return seconds > 0, {"logRateGovernorSeconds": round(seconds, 1)}
    if signal == "deadlock":
        rates_part = _slice("rates", target, start, end)
        if rates_part is None:
            return None, {}
        deadlocks = 0.0 if rates_part.empty else _safe_number(
            rates_part.loc[rates_part["CounterName"] == "Number of Deadlocks/sec", "Delta"].sum())
        return deadlocks > 0, {"deadlocks": deadlocks}
    if signal in WAIT_FAMILY_CATEGORIES:
        # A wait family: its category's waits average at least EPISODE_WAIT_SECONDS per bin in the window.
        waits_part = _slice("waits", target, start, end)
        if waits_part is None:
            return None, {}
        category = WAIT_FAMILY_CATEGORIES[signal]
        part = waits_part[waits_part["WaitCategory"] == category] if not waits_part.empty else waits_part
        per_bin = 0.0 if part.empty else _safe_number(part.groupby("Bucket")["WaitMs"].sum().mean()) / 1000.0
        return per_bin >= EPISODE_WAIT_SECONDS, {"category": category, "waitSecondsPerBin": round(per_bin, 1)}
    rates_part, sessions_part = _slice("rates", target, start, end), _slice("sessions", target, start, end)
    if rates_part is None and sessions_part is None:
        return None, {}
    batches = 0.0 if rates_part is None or rates_part.empty else _safe_number(
        rates_part.loc[rates_part["CounterName"] == "Batch Requests/sec", "Delta"].sum())
    max_sessions = 0.0 if sessions_part is None or sessions_part.empty else _safe_number(sessions_part["MaxSessions"].max())
    return batches > 0 or max_sessions > 0, {"batchRequests": batches, "maxSessions": max_sessions}


WINDOWS = scenario_windows(WINDOW_START, WINDOW_END)
incident_windows = [window for window in WINDOWS if window["profile"].startswith("incident:") or window["profile"] == "business-day"]
rows = []
for window in WINDOWS:
    if window["profile"] == "off":
        continue
    for target in window["targets"]:
        if target not in SELECTED_TARGETS:
            continue
        check_end = min(window["end"] + LAG, WINDOW_END)
        others = [item for item in incident_windows if item["request_id"] != window["request_id"] and target in item["targets"]]
        outcomes = {signal: evaluate_signal(signal, target, window["start"], check_end, others + [window])
                    for signal in expected_signals(window["profile"])}
        observed = [value for value, _ in outcomes.values()]
        status = "Pass" if any(value is True for value in observed) else (
            "Manual action required" if all(value is None for value in observed) else "Warning")
        evidence = {signal: details for signal, (_, details) in outcomes.items()}
        add_finding("correlation", target, status,
                    "%s on %s, %s to %s UTC: %s." % (window["profile"], target, window["start"].strftime("%H:%M"), window["end"].strftime("%H:%M"),
                                                     "expected signal observed" if status == "Pass" else "telemetry not queried here" if status != "Warning"
                                                     else "expected signal not observed (yet)"),
                    window["request_id"][:12], evidence=evidence, request_id=window["request_id"], start=window["start"], end=window["end"])
        rows.append({"Profile": window["profile"], "Target": target, "Start": window["start"].strftime("%H:%M"),
                     "End": window["end"].strftime("%H:%M"), "Status": status, "Evidence": json.dumps(evidence, default=str)})

if not WINDOWS:
    print("No scenario windows overlap the analysis window. Send a request in section 3, or widen WINDOW_MINUTES.")
elif rows:
    display_frame(pd.DataFrame(rows))

plot_panels("Scenario windows over the telemetry", [
    ("CPU, average per bin", to_series(FRAMES.get("cpu"), "AvgCPU", "max"), "% CPU"),
    ("Processes blocked (maximum per bin)", to_series(FRAMES.get("gauges"), "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked"), "processes"),
    ("Batch requests per bin", to_series(FRAMES.get("rates"), "Delta", "sum", lambda f: f["CounterName"] == "Batch Requests/sec"), "requests"),
], WINDOWS)

## 9. See it in the Hub

Open [Database Hub](https://powerbi.com/workloads/fdh/databaseHub) (Fabric Free license; it shows only what your Azure permissions allow):

1. **Overview:** **What needs attention?** shows issues and suggestions; every card leads to Estate. The CPU, memory, storage, and IO summaries should match the trends above.
2. **Estate:** set **View** to **Microsoft SQL**, filter to the demo resource group, and select the target that had the incident. Issues appear as colored chips, suggestions as lightbulb chips, and agent findings with a summary and a recommended action.
3. **Performance:** the **Microsoft SQL** dashboard. Select the same targets and time range as this notebook; check the resource selection before reading a chart as the whole estate.
4. **Database Agent and Copilot:** ask the prompts that the next cell prints, filled in with this session's targets and scenario window.

**T4, the PostgreSQL flexible server (Phase 24).** This notebook can't read T4: its tokens (`notebookutils`) cover storage, Power BI, Key Vault, and Kusto, not Azure Resource Manager or Azure Database for PostgreSQL, T4 has no public endpoint, and its metrics aren't in the telemetry endpoint above. Watch T4 live in Database Hub instead: in **Estate**, filter to **View: PostgreSQL** and select `pgflex01-<deploymentId>`, then open **Performance** and the **PostgreSQL** dashboard, which charts its Azure Monitor metrics (CPU, memory, storage, IOPS, connections, failed connections; [Monitor an Azure Database for PostgreSQL flexible server in the Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-postgresql)). Hub has no query-level diagnostics for PostgreSQL; T4's live sessions, waits, blockers, and connections against `max_connections` come from the runner: `./scripts/status.ps1 -Detail` (STATUS.T4 and STATUS.T4.LIVE) or `Invoke-DemoWorkloadCommand -Target T4 -ArgumentList status`.

**What to expect.** Describe the **signals**, never a specific answer: the agent is a preview, its output isn't deterministic, and its target coverage isn't documented. Any action it proposes needs a person to approve it, and nothing in this repo applies a fix automatically. If the agent doesn't cover a target, use the charts above, Query Store, or the `microsoft-sql` skills as the fallback.

In [ ]:
# Database Agent prompts to try, filled in with this session's names and latest scenario window. Prints text only.
names = {"T1": T1_VM_NAME, "T2": T2_MACHINE_NAME, "T3": T3_SERVER_PREFIX + "<id>/" + DEMO_DATABASE}
focus = next((window for window in reversed(WINDOWS) if window["profile"].startswith("incident:")), None)
focus_target = focus["targets"][0] if focus else "T1"
focus_name = names[focus_target]
if not focus:
    focus_time = "in the last 30 minutes"
elif (focus["end"] - focus["start"]).total_seconds() < 120:
    focus_time = "since %s UTC" % focus["start"].strftime("%H:%M")
else:
    focus_time = "between %s and %s UTC" % (focus["start"].strftime("%H:%M"), focus["end"].strftime("%H:%M"))
prompts = [
    ("Estate health", "Summarize the health of my SQL estate in resource group %s. What needs attention first?" % RESOURCE_GROUP,
     "every selected target named; issues and suggestions ranked"),
    ("Most pressure", "Which of my SQL resources is under the most pressure right now, and why?",
     "CPU, waits, or blocking tied to one target and a time window, matching sections 6 and 7"),
    ("Investigate", "Investigate what happened on %s %s." % (focus_name, focus_time),
     "the incident's signal in section 8's table: blocked processes, Lock waits, or deadlocks; CPU; or its wait family"),
    ("Correlate", "Correlate it with CPU, memory, IO, sessions, and recent changes.", "a timeline across several signals"),
    ("Root cause", "Explain the root cause in customer-friendly terms, with the evidence you used.", "plain language plus metrics and a time window"),
    ("Next action", "What's the safest next action? Don't apply it.", "a proposal that isn't executed and names what needs approval"),
    ("Baseline vs. incident", "Compare the baseline with the incident on %s." % focus_name, "before and after values for the same signals"),
    ("Approvals", "Which of these actions need approval?", "every change listed as needing a person to approve it"),
    ("Verify", "What should I verify in the Azure portal, SSMS, or VS Code?", "concrete places to confirm each finding"),
    ("Summary", "Write a concise incident summary.", "timeline, impact, cause, action, and verification"),
]
print("Database Hub: %s" % DATABASE_HUB_URL)
for title, prompt, signals in prompts:
    print("\n[%s]\n  Ask:        %s\n  Look for:   %s" % (title, prompt, signals))

## 10. Optional: Real-Time Dashboard

For a live view that refreshes on its own, or when the notebook can't query the endpoint directly, build a **Fabric Real-Time Dashboard** over the same telemetry. Both ways are UI steps; there's no documented API for Database Hub's template wizard, so this notebook only checks whether a dashboard exists.

- **From Database Hub:** **Performance** > **Create SQL monitors from template**: choose a template, select the demo databases, pick this workspace, and create a dashboard (or add a page to an existing one).
- **By hand:** create a Real-Time Dashboard in this workspace, add a data source with the URI `https://adx.centralus.arcdataservices.com/kusto/` and the database `ArcSqlTelemetry`, and paste the KQL from sections 5–7 into tiles. Each viewer sees only the resources their own Azure permissions allow; no stored Fabric connection is needed.

The next cell lists the workspace's Real-Time Dashboards with the Fabric REST API (**Items - List Items**, type `KQLDashboard`) and the notebook user's `pbi` token. It changes nothing.

In [ ]:
# Optional Real-Time Dashboard: lists KQLDashboard items in this workspace (read-only Fabric REST call).
dashboards, error = fabric_list_items(WORKSPACE_ID, "KQLDashboard")
if dashboards is None:
    add_finding("real_time_dashboard", "fabric", "Warning", "The workspace's Real-Time Dashboards couldn't be listed: %s" % error, "exists")
elif dashboards:
    add_finding("real_time_dashboard", "fabric", "Pass",
                "Real-Time Dashboards in this workspace: %s." % ", ".join(sorted(item.get("displayName", "?") for item in dashboards)), "exists",
                evidence={"count": len(dashboards)})
else:
    add_finding("real_time_dashboard", "fabric", "Manual action required",
                "No Real-Time Dashboard in this workspace yet. Optional: create one from Database Hub (Performance > Create SQL monitors "
                "from template) or by hand over %s, database %s." % (TELEMETRY_CONNECTION_URI, TELEMETRY_DATABASE), "exists")

## 11. Persist

The telemetry endpoint keeps up to 7 days. The next cell saves this run's analysis to `LH_DatabaseHub`, so the demo's history outlives that window:

| Table | Contents |
|---|---|
| `telemetry_snapshots` | Long-format results: analysis, target, metric, bin time, value, unit, and the analysis window |
| `analysis_findings` | Every finding with its status, evidence, scenario request, and window |
| `scenario_runs` | Already written by section 3 when a request was sent |
| `incident_windows` | One row per incident window: each recorded incident request with its expected-signal verdict, and each detected signal episode that no request explains (blocked processes above 0; CPU at or above `EPISODE_CPU_PERCENT` or `PAGEIOLATCH` waits at or above `EPISODE_PAGEIOLATCH_SECONDS` per bin, for `EPISODE_MIN_BINS` bins; Log Rate Governor waits above 0). Merged on its key, so re-running never duplicates one |

Every row also gets `report_date_utc`, the UTC date of its primary time. The cell then readies the five tables that the Power BI history report reads (`fabric/powerbi`): it creates any that are missing, adds and backfills `report_date_utc` once on tables written before it existed, merges the incident windows, and compacts any table with more than 200 files. Each write is a single Parquet file.

It writes lakehouse data only and changes no Azure or Fabric resources. Re-running it in the same session replaces this run's rows instead of duplicating them.

In [ ]:
# Persist: telemetry_snapshots and analysis_findings in the default lakehouse (lakehouse data only).
if FINDINGS:
    findings_view = pd.DataFrame(FINDINGS)[["status", "analysis", "target", "finding"]]
    display_frame(findings_view)
    counts = findings_view["status"].value_counts()
    print("Findings: " + ", ".join("%s %d" % (status, counts[status]) for status in STATUSES if status in counts))
if SAVE_RESULTS:
    write_delta("telemetry_snapshots", SNAPSHOT_ROWS)
    write_delta("analysis_findings", FINDINGS)
    REPORT_TABLES = prepare_report_tables()
else:
    print("SAVE_RESULTS is False, so nothing was saved.")

## 12. Stop and clean up

When the demo is over:

1. **Stop the activity:** the next cell sets the profile to `off` on every target through `demo-control.yml` (P9), or prints the link and command for it.
2. **Stop the compute:** `./scripts/stop-demo.ps1` (or the scheduled stop workflow) shuts the T2 guest down cleanly before deallocating the host, deallocates T1, removes Bastion, and lets T3 auto-pause. Nightly auto-shutdown is the backstop, and every profile also stops at its maximum duration.
3. **Reset for the next demo:** `./scripts/reset-demo.ps1` returns `ContosoCare` to its seed state and verifies that no demo transaction is open.
4. **Remove everything:** `./scripts/destroy.ps1` or the `destroy.yml` workflow (with approval) deletes the Arc resources and the resource group. The lakehouse tables keep the demo's history; `NB_DatabaseHub_Deployment` section 18 can drop them.

> **Resource-changing cell.** With both `CONFIRM = True` and `STOP_ACTIVITY = True`, the next cell triggers `demo-control.yml` with `scenario=off` for all targets and records it in `scenario_runs`. Otherwise it only prints a dry run, so **Run all** with `CONFIRM = True` starts a scenario in section 3 without stopping it here. You can also stop by setting `PROFILE = "off"` and sending it from section 3.

In [ ]:
# RESOURCE-CHANGING CELL: sets the activity profile to "off" on every target through demo-control.yml.
STOP_REQUEST = build_simulation_request("off", "all", 1, 0, 0, False, True)
if not (CONFIRM and STOP_ACTIVITY):
    print("Dry run: this cell needs both CONFIRM = True and STOP_ACTIVITY = True, so nothing was triggered or recorded. It would send:")
    print_simulation_plan(STOP_REQUEST)
else:
    LAST_REQUEST = submit_simulation_request(STOP_REQUEST) or LAST_REQUEST

## 13. Next steps

- **Next demo:** start with the `baseline` profile at least 30 minutes ahead (the scheduled warm-up does this), so Database Hub has a "before" picture, then run one `incident:<scenario>` at a time and walk through sections 5–9.
- **History:** the Power BI report `RPT_DatabaseHub_History` (`fabric/powerbi/README.md`) shows `telemetry_snapshots`, `analysis_findings`, `scenario_runs`, `incident_windows`, and `validation_results` from `LH_DatabaseHub`, beyond the telemetry's 7-day retention.
- **Validation:** run `NB_DatabaseHub_Deployment` for the deployment checks, or `./scripts/validate.ps1` from the repo for the full catalog, including the Azure-side checks this notebook can't do.

**References:** [Query performance monitoring telemetry](https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry) · [Monitor Microsoft SQL in Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-sql) · [NotebookUtils credentials](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials) · [Create a workflow dispatch event (GitHub REST API)](https://docs.github.com/rest/actions/workflows#create-a-workflow-dispatch-event) · [Items - List Items (Fabric REST API)](https://learn.microsoft.com/rest/api/fabric/core/items/list-items) · [Create a Real-Time Dashboard](https://learn.microsoft.com/fabric/real-time-intelligence/dashboard-real-time-create) · [Database Hub agent skills](https://github.com/microsoft/microsoft-sql/tree/main/plugins/microsoft-sql-fdh)